In [2]:
import sys
import subprocess
from pathlib import Path

print("=" * 75)
print("DIAGNÓSTICO Y EJECUCIÓN - DIRECCIONES CLIENTES")
print("=" * 75)

# 1. Rutas posibles del proyecto
rutas = [
    Path(r"D:\API_SUNAT\API_SUNAT\API_MAPS"),
    Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS"),
    Path(r"D:\API_SUNAT\API_MAPS"),
    Path(r"D:\API_MAPS"),
    Path.cwd(),
    Path.cwd() / "API_MAPS",
]

script = None

print("\n1. BUSCANDO SCRIPT...")

for carpeta in rutas:
    candidato = carpeta / "01_preparar_direcciones_clientes.py"

    if candidato.is_file():
        script = candidato
        print(f"ENCONTRADO: {script}")
        break
    else:
        print(f"No encontrado: {candidato}")

# 2. Comprobación
if script is None:
    print("\n" + "=" * 75)
    print("ERROR: NO SE ENCONTRÓ EL SCRIPT")
    print("=" * 75)
    print("Coloca 01_preparar_direcciones_clientes.py")
    print("en la carpeta API_MAPS.")
    print(f"\nCarpeta actual de Jupyter: {Path.cwd()}")

else:
    csv = script.parent.parent / "direcciones.csv"
    preparador = script.parent / "01_preparar_direcciones_masivo.py"

    print("\n2. VERIFICANDO ARCHIVOS...")
    print(f"Script: {'OK' if script.exists() else 'FALTA'}")
    print(f"CSV: {'OK' if csv.exists() else 'FALTA'}")
    print(f"Preparador original: {'OK' if preparador.exists() else 'FALTA'}")

    print(f"\nRuta CSV esperada: {csv}")

    # 3. Ejecución
    if csv.exists() and preparador.exists():

        print("\n3. EJECUTANDO PREPARACIÓN...")
        print("=" * 75)

        resultado = subprocess.run(
            [sys.executable, "-u", str(script)],
            cwd=str(script.parent),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace"
        )

        if resultado.stdout:
            print(resultado.stdout)

        if resultado.stderr:
            print("\nDETALLE DEL ERROR:")
            print(resultado.stderr)

        print("\nCódigo de salida:", resultado.returncode)

        if resultado.returncode == 0:
            print("\nPREPARACIÓN FINALIZADA CORRECTAMENTE")
        else:
            print("\nLA EJECUCIÓN PRESENTÓ UN ERROR")

    else:
        print("\nNo se ejecutó: faltan archivos necesarios.")

DIAGNÓSTICO Y EJECUCIÓN - DIRECCIONES CLIENTES

1. BUSCANDO SCRIPT...
No encontrado: D:\API_SUNAT\API_SUNAT\API_MAPS\01_preparar_direcciones_clientes.py
ENCONTRADO: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\01_preparar_direcciones_clientes.py

2. VERIFICANDO ARCHIVOS...
Script: OK
CSV: OK
Preparador original: OK

Ruta CSV esperada: D:\API_SUNAT_BK\API_SUNAT\direcciones.csv

3. EJECUTANDO PREPARACIÓN...
Registros originales: 284,345 | Con dirección: 159,459 | Sin dirección: 124,886
Destino (sin tocar la base antigua): D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones
Bloque 1: 5,582/159,459 direcciones preparadas
Bloque 2: 11,252/159,459 direcciones preparadas
Bloque 3: 16,848/159,459 direcciones preparadas
Bloque 4: 22,477/159,459 direcciones preparadas
Bloque 5: 28,200/159,459 direcciones preparadas
Bloque 6: 33,775/159,459 direcciones preparadas
Bloque 7: 39,358/159,459 direcciones preparadas
Bloque 8: 44,919/159,459 direcciones preparadas
Bloque 9: 50,565/159,459 direcciones preparad

In [3]:
from pathlib import Path
import sqlite3
import pandas as pd

CARPETA = Path(
    r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones"
)

print("=" * 75)
print("DIAGNÓSTICO - DIRECCIONES PREPARADAS")
print("=" * 75)

# 1. Archivos generados
print("\n1. ARCHIVOS GENERADOS")

if not CARPETA.exists():
    raise FileNotFoundError(f"No existe la carpeta: {CARPETA}")

for archivo in sorted(CARPETA.iterdir()):
    if archivo.is_file():
        mb = archivo.stat().st_size / (1024**2)
        print(f"{archivo.name}: {mb:,.2f} MB")

# 2. Revisar CSV
print("\n2. REVISIÓN DEL CSV")

csv = CARPETA / "direcciones_preparadas_clientes.csv"

if csv.exists():
    df = pd.read_csv(
        csv,
        sep=None,
        engine="python",
        dtype=str,
        encoding="utf-8-sig"
    )

    print(f"Total filas: {len(df):,}")
    print(f"Total columnas: {len(df.columns)}")
    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nPrimeras 5 direcciones:")
    display(df.head(5))

    print("\nValores nulos por columna:")
    print(df.isna().sum().to_string())
else:
    print("CSV preparado no encontrado.")

# 3. Revisar SQLite
print("\n3. BASE DE DATOS SQLITE")

db = CARPETA / "direcciones_preparadas_google_masivo.sqlite"

if db.exists():

    conexion = sqlite3.connect(str(db))

    tablas = pd.read_sql_query(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' ORDER BY name",
        conexion
    )

    print("\nTablas encontradas:")
    display(tablas)

    for tabla in tablas["name"]:
        nombre_seguro = tabla.replace('"', '""')

        cantidad = pd.read_sql_query(
            f'SELECT COUNT(*) AS total FROM "{nombre_seguro}"',
            conexion
        ).iloc[0]["total"]

        print(f"\nTABLA: {tabla}")
        print(f"Registros: {cantidad:,}")

        detalle = pd.read_sql_query(
            f'PRAGMA table_info("{nombre_seguro}")',
            conexion
        )

        display(detalle[["name", "type", "pk"]])

        muestra = pd.read_sql_query(
            f'SELECT * FROM "{nombre_seguro}" LIMIT 3',
            conexion
        )

        display(muestra)

    conexion.close()

else:
    print("Base SQLite no encontrada.")

print("\n" + "=" * 75)
print("DIAGNÓSTICO FINALIZADO")
print("=" * 75)

DIAGNÓSTICO - DIRECCIONES PREPARADAS

1. ARCHIVOS GENERADOS
diagnostico_clientes.csv: 0.00 MB
direcciones_preparadas_clientes.csv: 87.00 MB
direcciones_preparadas_google_masivo.sqlite: 104.53 MB
manifest_google_masivo.json: 0.00 MB

2. REVISIÓN DEL CSV
Total filas: 159,459
Total columnas: 65

Columnas:
['FILA_ID', 'BLOQUE_LAPTOP', 'ORDEN_LAPTOP', 'NumeroRuc', 'Nombre ó RazonSocial', 'Departamento', 'Provincia', 'Distrito', 'Direccion', 'DIRECCION_KEY', 'TIPO_DIRECCION', 'CALIDAD_INICIAL', 'DIRECCION_LIMPIA', 'REFERENCIA', 'TIPO_VIA', 'VIA_ORIGINAL', 'VIA_CORREGIDA', 'NUMERO_BUSCADO', 'MANZANA', 'LOTE', 'KM', 'ZONA_ORIGINAL', 'ZONA_CORREGIDA', 'HUBO_CORRECCION', 'Q1_TIPO', 'Q1', 'Q2_TIPO', 'Q2', 'Q3_TIPO', 'Q3', 'Q4_TIPO', 'Q4', 'Q5_TIPO', 'Q5', 'Q6_TIPO', 'Q6', 'Q7_TIPO', 'Q7', 'Q8_TIPO', 'Q8', 'Q9_TIPO', 'Q9', 'Q10_TIPO', 'Q10', 'Q11_TIPO', 'Q11', 'Q12_TIPO', 'Q12', 'Q13_TIPO', 'Q13', 'Q14_TIPO', 'Q14', 'Q15_TIPO', 'Q15', 'Q16_TIPO', 'Q16', 'Q17_TIPO', 'Q17', 'Q18_TIPO', 'Q18', 'perso

,FILA_ID,BLOQUE_LAPTOP,ORDEN_LAPTOP,NumeroRuc,Nombre ó RazonSocial,Departamento,Provincia,Distrito,Direccion,DIRECCION_KEY,...,Q16,Q17_TIPO,Q17,Q18_TIPO,Q18,personal_id,customer_id,taxpayer_id,ruc_id,FILA_ORIGINAL
0,1,1,1,10151048,NaN,LIMA,LIMA,CHACLACAYO,CAL HUASCATA MZ B CHACLACAYO,200aaca53deabc9efebc5e6552711a0fb14a0503,...,NaN,NaN,NaN,NaN,NaN,10151048,9016066,10101510480,NaN,2
1,2,1,2,10167090,NaN,LIMA,LIMA,LOS OLIVOS,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...,063b1910f16513872de756290a1ac7e51082bd80,...,NaN,NaN,NaN,NaN,NaN,10167090,NaN,10101670908,NaN,3
2,3,1,3,10247755,NaN,JUNÍN,CONCEPCIÓN,ORCOTUNA,JR PUNO URB ORCOTIUNA ORCOTUNA,88999b598a29144c909ed2b18e6f116c9da89d82,...,NaN,NaN,NaN,NaN,NaN,10247755,NaN,10102477559,NaN,4
3,4,1,4,1032902,NaN,SAN MARTIN,RIOJA,RIOJA,CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,6518ee40953835481ff82915200e938d49b9e468,...,NaN,NaN,NaN,NaN,NaN,1032902,20647830,10010329022,NaN,6
4,5,1,5,1052255,NaN,LAMBAYEQUE,FERREÑAFE,FERREÑAFE,AV BATANGRANDE 876 URB SAGRADO CORAZON DE JESU...,350f1448e47ac95fbeb33d8710c4a63e7ad4b2e0,...,NaN,NaN,NaN,NaN,NaN,1052255,21597817,10010522558,NaN,7



Valores nulos por columna:
FILA_ID                      0
BLOQUE_LAPTOP                0
ORDEN_LAPTOP                 0
NumeroRuc                    0
Nombre ó RazonSocial    159459
Departamento              1040
Provincia                 1040
Distrito                  1040
Direccion                    0
DIRECCION_KEY                0
TIPO_DIRECCION               0
CALIDAD_INICIAL              0
DIRECCION_LIMPIA             0
REFERENCIA              159459
TIPO_VIA                 24093
VIA_ORIGINAL             24094
VIA_CORREGIDA            24093
NUMERO_BUSCADO          159065
MANZANA                 159160
LOTE                    155546
KM                      158332
ZONA_ORIGINAL            50173
ZONA_CORREGIDA           50173
HUBO_CORRECCION              0
Q1_TIPO                      0
Q1                           0
Q2_TIPO                   6078
Q2                        6078
Q3_TIPO                  66973
Q3                       66973
Q4_TIPO                  68572
Q4         

,name
0,preparadas



TABLA: preparadas
Registros: 159,459


,name,type,pk
0,FILA_ID,INTEGER,0
1,BLOQUE_LAPTOP,INTEGER,0
2,ORDEN_LAPTOP,INTEGER,0
3,NumeroRuc,TEXT,0
4,Nombre ó RazonSocial,TEXT,0
...,...,...,...
60,personal_id,TEXT,0
61,customer_id,TEXT,0
62,taxpayer_id,TEXT,0
63,ruc_id,TEXT,0


,FILA_ID,BLOQUE_LAPTOP,ORDEN_LAPTOP,NumeroRuc,Nombre ó RazonSocial,Departamento,Provincia,Distrito,Direccion,DIRECCION_KEY,...,Q16,Q17_TIPO,Q17,Q18_TIPO,Q18,personal_id,customer_id,taxpayer_id,ruc_id,FILA_ORIGINAL
0,1,1,1,10151048,,LIMA,LIMA,CHACLACAYO,CAL HUASCATA MZ B CHACLACAYO,200aaca53deabc9efebc5e6552711a0fb14a0503,...,,,,,,10151048,9016066,10101510480,,2
1,2,1,2,10167090,,LIMA,LIMA,LOS OLIVOS,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...,063b1910f16513872de756290a1ac7e51082bd80,...,,,,,,10167090,,10101670908,,3
2,3,1,3,10247755,,JUNÍN,CONCEPCIÓN,ORCOTUNA,JR PUNO URB ORCOTIUNA ORCOTUNA,88999b598a29144c909ed2b18e6f116c9da89d82,...,,,,,,10247755,,10102477559,,4



DIAGNÓSTICO FINALIZADO


In [6]:
from pathlib import Path
import subprocess
import sys
import sqlite3
import pandas as pd
import re
import shutil
from datetime import datetime

print("=" * 85)
print("CORRECCION Y EJECUCION - GOOGLE MAPS CLIENTES V2")
print("=" * 85)

# ============================================================
# 1. RUTAS DEL PROYECTO
# ============================================================

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")

SCRIPT = BASE / "02_google_maps_clientes_v2.py"

CARPETA = BASE / "clientes_direcciones"

DB_PREPARADAS = (
    CARPETA / "direcciones_preparadas_google_masivo.sqlite"
)

DB_RESULTADOS = (
    CARPETA / "google_resultados_laptop1_v2.sqlite"
)

print("\n1. VERIFICANDO ARCHIVOS")

for nombre, ruta in [
    ("Robot Python", SCRIPT),
    ("Base SQLite preparada", DB_PREPARADAS),
]:
    print(f"{nombre}: {'OK' if ruta.is_file() else 'NO EXISTE'}")
    print(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(ruta)

# ============================================================
# 2. COPIA DE SEGURIDAD
# ============================================================

print("\n2. CREANDO RESPALDO DEL ROBOT")

fecha = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP = BASE / f"02_google_maps_clientes_v2_backup_{fecha}.py"

shutil.copy2(SCRIPT, BACKUP)

print("Respaldo:", BACKUP)

# ============================================================
# 3. CORREGIR CONFIGURACION
# ============================================================

print("\n3. CORRIGIENDO RUTAS")

codigo = SCRIPT.read_text(encoding="utf-8-sig")

configuracion = {
    "BASE_DIR": 'Path(__file__).resolve().parent',
    "OUT_DIR": 'BASE_DIR / "clientes_direcciones"',
    "PREP_DB": 'OUT_DIR / "direcciones_preparadas_google_masivo.sqlite"',
    "MANIFEST": 'OUT_DIR / "manifest_google_masivo.json"',
}

for variable, valor in configuracion.items():

    patron = rf"(?m)^{variable}\s*=.*$"

    reemplazo = f"{variable} = {valor}"

    codigo, cantidad = re.subn(
        patron,
        lambda m, r=reemplazo: r,
        codigo,
        count=1
    )

    if cantidad != 1:
        raise RuntimeError(
            f"No se encontro la configuracion: {variable}"
        )

    print(f"{variable}: CORREGIDO")

# ============================================================
# 4. VALIDAR SINTAXIS Y GUARDAR
# ============================================================

print("\n4. VALIDANDO SINTAXIS PYTHON")

try:
    compile(codigo, str(SCRIPT), "exec")
except SyntaxError as e:
    print("ERROR DE SINTAXIS:", e)
    print("El archivo original no fue modificado.")
    raise

SCRIPT.write_text(codigo, encoding="utf-8")

print("Sintaxis: OK")
print("Robot actualizado correctamente.")

# ============================================================
# 5. VALIDAR ESTRUCTURA DE SQLITE
# ============================================================

print("\n5. VERIFICANDO BASE PREPARADA")

with sqlite3.connect(DB_PREPARADAS) as conn:

    total = conn.execute(
        "SELECT COUNT(*) FROM preparadas"
    ).fetchone()[0]

    columnas = [
        fila[1]
        for fila in conn.execute(
            "PRAGMA table_info(preparadas)"
        ).fetchall()
    ]

print(f"Total direcciones: {total:,}")
print(f"Columnas: {len(columnas)}")

for campo in ["FILA_ID", "BLOQUE_LAPTOP", "Q1", "Direccion"]:
    if campo not in columnas:
        raise RuntimeError(
            f"Falta columna obligatoria: {campo}"
        )

print("Estructura SQLite: OK")

# ============================================================
# 6. EJECUTAR GOOGLE MAPS - 20 DIRECCIONES
# ============================================================

print("\n" + "=" * 85)
print("INICIANDO GOOGLE MAPS - PRIMERAS 20 DIRECCIONES")
print("=" * 85)

comando = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--laptop", "1",
    "--max-filas", "20"
]

# Se muestra la salida en vivo.
# Esto permite ver el avance y posibles errores.

resultado = subprocess.run(
    comando,
    cwd=str(BASE),
    check=False
)

print("\nCODIGO DE SALIDA:", resultado.returncode)

# ============================================================
# 7. ANALIZAR RESULTADOS
# ============================================================

print("\n" + "=" * 85)
print("DIAGNOSTICO DE RESULTADOS")
print("=" * 85)

if DB_RESULTADOS.is_file():

    with sqlite3.connect(DB_RESULTADOS) as conn:

        tablas = [
            r[0]
            for r in conn.execute(
                "SELECT name FROM sqlite_master "
                "WHERE type='table'"
            ).fetchall()
        ]

        if "resultados" in tablas:

            resumen = pd.read_sql_query(
                """
                SELECT
                    ESTADO,
                    COUNT(*) AS CANTIDAD
                FROM resultados
                GROUP BY ESTADO
                ORDER BY CANTIDAD DESC
                """,
                conn
            )

            print("\nRESUMEN POR ESTADO")
            display(resumen)

            detalle = pd.read_sql_query(
                """
                SELECT
                    FILA_ID,
                    DIRECCION_ORIGINAL,
                    Distrito,
                    ESTADO,
                    NIVEL_UBICACION_GOOGLE,
                    CONFIANZA_GOOGLE,
                    LATITUD,
                    LONGITUD,
                    QUERY_USADA
                FROM resultados
                ORDER BY FILA_ID
                LIMIT 20
                """,
                conn
            )

            print("\nDETALLE DE DIRECCIONES")
            display(detalle)

        else:
            print("La tabla resultados aun no se ha creado.")

else:
    print("Todavia no existe la base de resultados.")

print("\n" + "=" * 85)
print("PROCESO FINALIZADO")
print("=" * 85)

CORRECCION Y EJECUCION - GOOGLE MAPS CLIENTES V2

1. VERIFICANDO ARCHIVOS
Robot Python: OK
D:\API_SUNAT_BK\API_SUNAT\API_MAPS\02_google_maps_clientes_v2.py
Base SQLite preparada: OK
D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\direcciones_preparadas_google_masivo.sqlite

2. CREANDO RESPALDO DEL ROBOT
Respaldo: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\02_google_maps_clientes_v2_backup_20261116_151031.py

3. CORRIGIENDO RUTAS
BASE_DIR: CORREGIDO
OUT_DIR: CORREGIDO
PREP_DB: CORREGIDO
MANIFEST: CORREGIDO

4. VALIDANDO SINTAXIS PYTHON
Sintaxis: OK
Robot actualizado correctamente.

5. VERIFICANDO BASE PREPARADA
Total direcciones: 159,459
Columnas: 65
Estructura SQLite: OK

INICIANDO GOOGLE MAPS - PRIMERAS 20 DIRECCIONES

CODIGO DE SALIDA: 0

DIAGNOSTICO DE RESULTADOS

RESUMEN POR ESTADO


,ESTADO,CANTIDAD
0,ENCONTRADO_REFERENCIA,20



DETALLE DE DIRECCIONES


,FILA_ID,DIRECCION_ORIGINAL,Distrito,ESTADO,NIVEL_UBICACION_GOOGLE,CONFIANZA_GOOGLE,LATITUD,LONGITUD,QUERY_USADA
0,1,CAL HUASCATA MZ B CHACLACAYO,CHACLACAYO,ENCONTRADO_REFERENCIA,VIA,MEDIA,-11.994911,-76.820873,"CAL HUASCATA MZ B CHACLACAYO, CHACLACAYO, LIMA..."
1,2,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...,LOS OLIVOS,ENCONTRADO_REFERENCIA,ZONA,MEDIA,-11.973470,-77.073291,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...
2,3,JR PUNO URB ORCOTIUNA ORCOTUNA,ORCOTUNA,ENCONTRADO_REFERENCIA,ZONA,MEDIA,-11.966760,-75.329557,"JR PUNO URB ORCOTIUNA ORCOTUNA, ORCOTUNA, CONC..."
3,4,CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,RIOJA,ENCONTRADO_REFERENCIA,DISTRITO,BAJA,-6.062355,-77.168126,"RIOJA, SAN MARTIN, PERU"
4,5,AV BATANGRANDE 876 URB SAGRADO CORAZON DE JESU...,FERREÑAFE,ENCONTRADO_REFERENCIA,VIA,MEDIA,-6.625085,-79.786210,"AVENIDA BATANGRANDE 876, FERRENAFE, LAMBAYEQUE..."
5,6,JR VILCANOTA MZ 10 URB ACHO MEGO YARINACOCHA,YARINACOCHA,ENCONTRADO_REFERENCIA,ZONA,MEDIA,-8.388753,-74.584367,"ACHO MEGO YARINACOCHA, YARINACOCHA, CORONEL PO..."
6,7,MZ 7 URB HORACIO ZEBALLOS GAMEZ SOCABAYA,SOCABAYA,ENCONTRADO_REFERENCIA,ZONA,MEDIA,-16.489928,-71.506160,"MZ 7 URB HORACIO ZEBALLOS GAMEZ SOCABAYA, SOCA..."
7,8,AV CARRETERA CENTRAL 405 ATE,ATE,ENCONTRADO_REFERENCIA,ZONA,MEDIA,-12.010324,-76.871470,"AV CARRETERA CENTRAL 405 ATE, ATE, LIMA, PERU"
8,9,AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO,TARAPOTO,ENCONTRADO_REFERENCIA,VIA_ZONA,MEDIA_ALTA,-6.475129,-76.372398,"AVENIDA CIRCUNVALACION 2135, TARAPOTO TARAPOTO..."
9,10,CAL LA LIBERTAD SECTOR CAMPO SERIAL 100 URB VA...,PICHARI,ENCONTRADO_REFERENCIA,VIA_ZONA,MEDIA_ALTA,-12.505988,-73.821852,CAL LA LIBERTAD SECTOR CAMPO SERIAL 100 URB VA...



PROCESO FINALIZADO


In [7]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE = Path(
    r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones"
)

DB = BASE / "google_resultados_laptop1_v2.sqlite"

FILAS_REVISAR = [4, 9, 13, 16, 19, 20]

if not DB.is_file():
    raise FileNotFoundError(DB)

with sqlite3.connect(DB) as conn:

    ids = ",".join(str(x) for x in FILAS_REVISAR)

    seleccionados = pd.read_sql_query(
        f"""
        SELECT
            FILA_ID,
            DIRECCION_ORIGINAL,
            Distrito,
            ESTADO,
            NIVEL_UBICACION_GOOGLE,
            SCORE_GOOGLE,
            LATITUD,
            LONGITUD,
            FUENTE_COORDENADAS,
            VIA_SIM,
            DISTRITO_COV,
            FLAG_NUMERO_OK,
            NOMBRE_GOOGLE,
            DIRECCION_GOOGLE,
            QUERY_USADA
        FROM resultados
        WHERE FILA_ID IN ({ids})
        ORDER BY FILA_ID
        """,
        conn
    )

    candidatos = pd.read_sql_query(
        f"""
        SELECT
            FILA_ID,
            QUERY_TIPO,
            QUERY,
            SCORE,
            NIVEL,
            LATITUD,
            LONGITUD,
            VIA_SIM,
            DISTRITO_COV,
            NOMBRE_GOOGLE,
            DIRECCION_GOOGLE
        FROM candidatos
        WHERE FILA_ID IN ({ids})
        ORDER BY FILA_ID, SCORE DESC
        """,
        conn
    )

print("=" * 80)
print("RESULTADOS SELECCIONADOS POR EL ROBOT")
print("=" * 80)

display(seleccionados)

print("\n" + "=" * 80)
print("MEJORES CANDIDATOS ALTERNATIVOS")
print("=" * 80)

for fila in FILAS_REVISAR:
    muestra = candidatos[
        candidatos["FILA_ID"] == fila
    ].head(5)

    print(f"\nFILA {fila}")
    display(muestra)

print("\nDIAGNOSTICO FINALIZADO")

RESULTADOS SELECCIONADOS POR EL ROBOT


,FILA_ID,DIRECCION_ORIGINAL,Distrito,ESTADO,NIVEL_UBICACION_GOOGLE,SCORE_GOOGLE,LATITUD,LONGITUD,FUENTE_COORDENADAS,VIA_SIM,DISTRITO_COV,FLAG_NUMERO_OK,NOMBRE_GOOGLE,DIRECCION_GOOGLE,QUERY_USADA
0,4,CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,RIOJA,ENCONTRADO_REFERENCIA,DISTRITO,33.00,-6.062355,-77.168126,FICHA_URL_3D4D,0.0,1.0,0,Rioja,,"RIOJA, SAN MARTIN, PERU"
1,9,AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO,TARAPOTO,ENCONTRADO_REFERENCIA,VIA_ZONA,83.00,-6.475129,-76.372398,FICHA_URL_3D4D,1.0,1.0,0,Av. Circunvalación 2135,,"AVENIDA CIRCUNVALACION 2135, TARAPOTO TARAPOTO..."
2,13,CAL MZ F5 LT 33 URB BRISAS OQUENDO MZ F5 URB U...,CALLAO,ENCONTRADO_REFERENCIA,VIA,52.09,-12.142464,-76.961559,FICHA_URL_3D4D,1.0,0.0,0,Mz F5,,"CALLE MZ F5, LT 33, URB LAS BRISAS DE OQUENDO ..."
3,16,CAL XDEPURAR 88888888,LIMA,ENCONTRADO_REFERENCIA,DISTRITO,33.00,-12.046689,-77.043089,FICHA_URL_3D4D,0.0,1.0,0,Lima,,"LIMA, PERU"
4,19,CAL MZ B URB ANDAHUASI SAYAN,SAYÁN,ENCONTRADO_REFERENCIA,VIA,51.25,-12.098778,-76.962168,FICHA_URL_3D4D,1.0,0.0,0,Mz B,,"CALLE MZ B, SAYAN, HUAURA, LIMA, PERU"
5,20,AV BENJAMIN DOIG LOSSIO 396 CERCADO DE LIMA,LIMA,ENCONTRADO_REFERENCIA,ZONA,25.50,-12.097548,-76.964951,FICHA_URL_3D4D,0.0,1.0,0,Ca. Benjamín Doig Lossio 396,,"AV BENJAMIN DOIG LOSSIO 396 CERCADO DE LIMA, L..."



MEJORES CANDIDATOS ALTERNATIVOS

FILA 4


,FILA_ID,QUERY_TIPO,QUERY,SCORE,NIVEL,LATITUD,LONGITUD,VIA_SIM,DISTRITO_COV,NOMBRE_GOOGLE,DIRECCION_GOOGLE
0,4,ORIGINAL,"CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,...",38.83,SIN_MATCH,-6.057048,-77.165772,0.167,1.0,Barrio quinapata,"WRVM+5MQ, Rioja 22826, Perú"
1,4,SOLO_VIA,CALLE ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJ...,38.83,SIN_MATCH,-6.057048,-77.165772,0.167,1.0,Barrio quinapata,"WRVM+5MQ, Rioja 22826, Perú"
2,4,DISTRITO_FALLBACK,"RIOJA, SAN MARTIN, PERU",33.00,DISTRITO,-6.062355,-77.168126,0.000,1.0,Rioja,
3,4,ORIGINAL,"CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,...",15.00,SIN_MATCH,-6.062914,-77.171741,0.000,0.0,Almte. Grau,
4,4,SOLO_VIA,CALLE ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJ...,15.00,SIN_MATCH,-6.062914,-77.171741,0.000,0.0,Almte. Grau,



FILA 9


,FILA_ID,QUERY_TIPO,QUERY,SCORE,NIVEL,LATITUD,LONGITUD,VIA_SIM,DISTRITO_COV,NOMBRE_GOOGLE,DIRECCION_GOOGLE
6,9,VIA_ZONA,"AVENIDA CIRCUNVALACION 2135, TARAPOTO TARAPOTO...",83.0,VIA_ZONA,-6.475129,-76.372398,1.0,1.0,Av. Circunvalación 2135,
7,9,SOLO_VIA,"AVENIDA CIRCUNVALACION 2135, TARAPOTO, SAN MAR...",83.0,VIA_ZONA,-6.475129,-76.372398,1.0,1.0,Av. Circunvalación 2135,
8,9,ORIGINAL,"AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO, ...",48.0,ZONA,-6.492995,-76.373649,0.0,1.0,Empresa San Martín S.A.,"Av Alfonso Ugarte 1480, Tarapoto 22201, Perú"
9,9,ORIGINAL,"AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO, ...",48.0,ZONA,-6.508837,-76.365497,0.0,1.0,Motos SSenda Tarapoto,"Via de Evitamiento 2060, Tarapoto 22201, Perú"
10,9,ORIGINAL,"AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO, ...",48.0,ZONA,-6.492879,-76.364722,0.0,1.0,Edipesa Tarapoto,"Jirón Jimenez Pimentel 838, Tarapoto 22201, Perú"


... (5 outputs left out)

,FILA_ID,QUERY_TIPO,QUERY,SCORE,NIVEL,LATITUD,LONGITUD,VIA_SIM,DISTRITO_COV,NOMBRE_GOOGLE,DIRECCION_GOOGLE
27,19,SOLO_VIA,"CALLE MZ B, SAYAN, HUAURA, LIMA, PERU",51.25,VIA,-12.098778,-76.962168,1.0,0.0,Mz B,
28,19,SOLO_VIA,"CALLE MZ B, SAYAN, HUAURA, LIMA, PERU",48.00,ZONA,-11.137463,-77.197400,0.0,1.0,Sayán,
29,19,ORIGINAL,"CAL MZ B URB ANDAHUASI SAYAN, SAYAN, HUAURA, L...",25.00,ZONA,-11.149873,-77.244107,0.0,0.0,Andahuasi,
30,19,VIA_ZONA,"CALLE MZ B, ANDAHUASI SAYAN, SAYAN, HUAURA, LI...",25.00,ZONA,-11.149873,-77.244107,0.0,0.0,Andahuasi,
31,19,SOLO_ZONA,"ANDAHUASI SAYAN, SAYAN, HUAURA, LIMA, PERU",25.00,ZONA,-11.149873,-77.244107,0.0,0.0,Andahuasi,



FILA 20


,FILA_ID,QUERY_TIPO,QUERY,SCORE,NIVEL,LATITUD,LONGITUD,VIA_SIM,DISTRITO_COV,NOMBRE_GOOGLE,DIRECCION_GOOGLE
32,20,ORIGINAL,"AV BENJAMIN DOIG LOSSIO 396 CERCADO DE LIMA, L...",25.5,ZONA,-12.097548,-76.964951,0.0,1.0,Ca. Benjamín Doig Lossio 396,
33,20,VIA_ZONA,AVENIDA BENJAMIN DOIG LOSSIO 396 CERCADO DE LI...,25.5,ZONA,-12.097548,-76.964951,0.0,1.0,Ca. Benjamín Doig Lossio 396,
34,20,SOLO_VIA,AVENIDA BENJAMIN DOIG LOSSIO 396 CERCADO DE LI...,25.5,ZONA,-12.097548,-76.964951,0.0,1.0,Ca. Benjamín Doig Lossio 396,
35,20,SOLO_ZONA,"BENJAMIN DOIG LOSSIO 396 CERCADO DE LIMA, LIMA...",25.5,ZONA,-12.097548,-76.964951,0.0,1.0,Ca. Benjamín Doig Lossio 396,



DIAGNOSTICO FINALIZADO


In [8]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
SCRIPT = BASE / "03_auditar_candidatos_clientes.py"
SALIDA = BASE / "clientes_direcciones"

if not SCRIPT.is_file():
    raise FileNotFoundError(f"No se encontró: {SCRIPT}")

resultado = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(resultado.stdout)

if resultado.stderr:
    print("ERRORES:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        f"Auditoría fallida: código {resultado.returncode}"
    )

archivo = SALIDA / "auditoria_coordenadas_clientes.csv"

df_auditoria = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8-sig"
)

print("\nRESUMEN DE LA AUDITORÍA")
display(
    df_auditoria["DECISION_PRELIMINAR"]
    .value_counts()
    .reset_index()
)

print("\nDETALLE DE LOS RESULTADOS")
display(
    df_auditoria[
        [
            "FILA_ID",
            "DIRECCION_ORIGINAL",
            "LATITUD_ORIGINAL",
            "LONGITUD_ORIGINAL",
            "LATITUD_PROPUESTA",
            "LONGITUD_PROPUESTA",
            "CLASIFICACION_PROPUESTA",
            "DISTANCIA_CAMBIO_KM",
            "DECISION_PRELIMINAR"
        ]
    ]
)

AUDITORIA PRELIMINAR DE GEOLOCALIZACION - SIN CONSULTAR GOOGLE Y SIN MODIFICAR SQLITE
Filas evaluadas: 20 | Candidatos conservados: 131

RESUMEN DE DECISIONES:
DECISION_PRELIMINAR
COINCIDE_O_MUY_PROXIMA            11
REVISAR_SIN_ALTERNATIVA_SEGURA     4
ALTERNATIVA_DIFIERE_MAS_2KM        3
ALTERNATIVA_DIFERENTE_REVISAR      1
DIRECCION_INVALIDA                 1

CASOS IMPORTANTES:
 FILA_ID Distrito NIVEL_ORIGINAL         NOMBRE_CANDIDATO_PROPUESTO        CLASIFICACION_PROPUESTA  DISTANCIA_CAMBIO_KM           DECISION_PRELIMINAR
       4    RIOJA       DISTRITO                   Barrio quinapata    REFERENCIA_ZONA_SIN_VALIDAR                 0.64 ALTERNATIVA_DIFERENTE_REVISAR
       9 TARAPOTO       VIA_ZONA            Av. Circunvalación 2135 NUMERO_COINCIDENTE_SIN_VALIDAR                 0.00        COINCIDE_O_MUY_PROXIMA
      13   CALLAO            VIA Urbanización Las Brisas de Oquendo    REFERENCIA_ZONA_SIN_VALIDAR                27.02   ALTERNATIVA_DIFIERE_MAS_2KM
      16     LI

,DECISION_PRELIMINAR,count
0,COINCIDE_O_MUY_PROXIMA,11
1,REVISAR_SIN_ALTERNATIVA_SEGURA,4
2,ALTERNATIVA_DIFIERE_MAS_2KM,3
3,ALTERNATIVA_DIFERENTE_REVISAR,1
4,DIRECCION_INVALIDA,1



DETALLE DE LOS RESULTADOS


,FILA_ID,DIRECCION_ORIGINAL,LATITUD_ORIGINAL,LONGITUD_ORIGINAL,LATITUD_PROPUESTA,LONGITUD_PROPUESTA,CLASIFICACION_PROPUESTA,DISTANCIA_CAMBIO_KM,DECISION_PRELIMINAR
0,1,CAL HUASCATA MZ B CHACLACAYO,-11.994911,-76.820873,-11.994135,-76.821341,REFERENCIA_VIA_SIN_VALIDAR,0.10,COINCIDE_O_MUY_PROXIMA
1,2,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...,-11.973470,-77.073291,-11.973470,-77.073291,REFERENCIA_ZONA_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
2,3,JR PUNO URB ORCOTIUNA ORCOTUNA,-11.966760,-75.329557,NaN,NaN,NaN,NaN,REVISAR_SIN_ALTERNATIVA_SEGURA
3,4,CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,-6.062355,-77.168126,-6.057048,-77.165772,REFERENCIA_ZONA_SIN_VALIDAR,0.64,ALTERNATIVA_DIFERENTE_REVISAR
4,5,AV BATANGRANDE 876 URB SAGRADO CORAZON DE JESU...,-6.625085,-79.786210,-6.625085,-79.786210,NUMERO_COINCIDENTE_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
5,6,JR VILCANOTA MZ 10 URB ACHO MEGO YARINACOCHA,-8.388753,-74.584367,-8.388753,-74.584367,REFERENCIA_ZONA_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
6,7,MZ 7 URB HORACIO ZEBALLOS GAMEZ SOCABAYA,-16.489928,-71.506160,-16.489928,-71.506160,REFERENCIA_ZONA_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
7,8,AV CARRETERA CENTRAL 405 ATE,-12.010324,-76.871470,-12.010324,-76.871470,NUMERO_COINCIDENTE_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
8,9,AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO,-6.475129,-76.372398,-6.475129,-76.372398,NUMERO_COINCIDENTE_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA
9,10,CAL LA LIBERTAD SECTOR CAMPO SERIAL 100 URB VA...,-12.505988,-73.821852,-12.505988,-73.821852,REFERENCIA_ZONA_SIN_VALIDAR,0.00,COINCIDE_O_MUY_PROXIMA


In [10]:
from pathlib import Path
import subprocess
import sys

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")

SCRIPT = BASE / "04_validar_territorio_clientes.py"
CAPAS = BASE / "clientes_direcciones" / "limites_inei"

print("=" * 80)
print("PASO 4 - VALIDACION TERRITORIAL DE CLIENTES")
print("=" * 80)

CAPAS.mkdir(parents=True, exist_ok=True)

if not SCRIPT.exists():
    raise FileNotFoundError(f"Falta el script: {SCRIPT}")

archivos = [
    p for p in CAPAS.rglob("*")
    if p.suffix.lower() in (".gpkg", ".shp", ".zip")
]

if not archivos:
    print("No se encontro una capa geografica.")
    print("Descarga y descomprime los limites distritales del INEI.")
    print("Carpeta de destino:", CAPAS)

else:
    print("Capas encontradas:")
    for p in archivos:
        print(" -", p.name)

    print("\nIniciando validacion...\n")

    resultado = subprocess.run(
        [sys.executable, "-u", str(SCRIPT)],
        cwd=str(BASE),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False
    )

    print(resultado.stdout)

    if resultado.stderr:
        print("\nDETALLE DEL ERROR:")
        print(resultado.stderr)

    print("Codigo de salida:", resultado.returncode)

PASO 4 - VALIDACION TERRITORIAL DE CLIENTES
Capas encontradas:
 - DISTRITO.gpkg

Iniciando validacion...

VALIDACION TERRITORIAL DE COORDENADAS - MODO SOLO LECTURA
Archivo geográfico: DISTRITO.gpkg | Registros de límites: 1,890 | CRS: EPSG:4326
Campos de límites reconocidos: {'Departamento': 'nombdep', 'Provincia': 'nombprov', 'Distrito': 'nombdist'}
Divisiones territoriales disponibles: 1,890

ESTADO TERRITORIAL DE LOS CANDIDATOS (incluye rechazados textuales):
ESTADO_TERRITORIAL
DENTRO_DISTRITO           106
FUERA_DISTRITO_REVISAR     25

ESTADO DE LAS DIRECCIONES:
DECISION_TERRITORIAL
CANDIDATO_EN_DISTRITO_NO_PREDIAL              13
ORIGINAL_EN_DISTRITO_SIN_EVIDENCIA_TEXTUAL     5
DIRECCION_INVALIDA_NO_USAR                     1
SIN_CANDIDATO_CONFIABLE_EN_DISTRITO            1

CASOS CLAVE:
 FILA_ID Distrito    TERRITORIO_ORIGINAL TERRITORIO_PROPUESTA_PREVIA       NOMBRE_CANDIDATO_TERRITORIAL                DECISION_TERRITORIAL
       4    RIOJA        DENTRO_DISTRITO             DE

In [11]:
from pathlib import Path
import subprocess
import sys

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")

SCRIPT = BASE / "05_consolidar_piloto_clientes.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)

resultado = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(resultado.stdout)

if resultado.stderr:
    print("DETALLE DEL ERROR:")
    print(resultado.stderr)

print("CODIGO DE SALIDA:", resultado.returncode)

PASO 5 - CONSOLIDACION DE PILOTO CLIENTES (SIN SOBREESCRIBIR FUENTES)
Direcciones del piloto: 20
Referencias territoriales provisionales: 13
Predios certificados: 0

CLASIFICACION:
                          CALIDAD_GEO                USO_GEO  DIRECCIONES  CON_COORDENADA_PROVISIONAL
                      REFERENCIA_ZONA REFERENCIA_PROVISIONAL            7                           7
COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR REFERENCIA_PROVISIONAL            5                           5
       REVISION_SIN_EVIDENCIA_TEXTUAL                NO_USAR            5                           0
                   DIRECCION_INVALIDA                NO_USAR            1                           0
                       REFERENCIA_VIA REFERENCIA_PROVISIONAL            1                           1
              SIN_CANDIDATO_CONFIABLE                NO_USAR            1                           0

CASOS PARA VERIFICAR:
 FILA_ID Distrito                                                                 

In [12]:
from pathlib import Path
import subprocess
import sys
import sqlite3
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
SCRIPT = BASE / "06_google_maps_clientes_v3.py"
CARPETA = BASE / "clientes_direcciones"
DB_V3 = CARPETA / "google_resultados_laptop1_v3.sqlite"
LOG = CARPETA / "ejecucion_piloto_v3.log"

print("=" * 85)
print("GOOGLE MAPS CLIENTES V3 - PRUEBA CONTROLADA")
print("=" * 85)

# 1. Verificación de archivos
if not SCRIPT.is_file():
    raise FileNotFoundError(f"Falta el robot: {SCRIPT}")

if not (
    CARPETA / "direcciones_preparadas_google_masivo.sqlite"
).is_file():
    raise FileNotFoundError("No existe la base preparada")

# 2. Validación de configuración
print("\n1. VERIFICANDO CONFIGURACIÓN Y POLÍGONOS INEI")

verificacion = subprocess.run(
    [sys.executable, "-u", str(SCRIPT),
     "--laptop", "1", "--verificar"],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(verificacion.stdout)

if verificacion.returncode != 0:
    print(verificacion.stderr)
    raise RuntimeError("Falló la validación inicial")

# 3. Ejecutar las primeras 20 direcciones
print("\n2. INICIANDO PRUEBA DE 20 DIRECCIONES")
print("Los resultados se guardarán en una base V3 independiente.\n")

comando = [
    sys.executable, "-u", str(SCRIPT),
    "--laptop", "1",
    "--desde", "1",
    "--hasta", "20",
    "--max-filas", "20"
]

with LOG.open("w", encoding="utf-8") as archivo_log:
    with subprocess.Popen(
        comando,
        cwd=str(BASE),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1
    ) as proceso:

        for linea in proceso.stdout:
            print(linea, end="", flush=True)
            archivo_log.write(linea)
            archivo_log.flush()

        codigo = proceso.wait()

print("\nCódigo de salida:", codigo)
print("Log completo:", LOG)

# 4. Consultar la base V3
if DB_V3.is_file():

    with sqlite3.connect(DB_V3) as conn:

        resumen = pd.read_sql_query(
            """
            SELECT
                ESTADO,
                CALIDAD_GEO,
                USO_GEO,
                COUNT(*) AS DIRECCIONES
            FROM resultados
            GROUP BY ESTADO, CALIDAD_GEO, USO_GEO
            ORDER BY DIRECCIONES DESC
            """,
            conn
        )

        detalle = pd.read_sql_query(
            """
            SELECT
                FILA_ID,
                DIRECCION_ORIGINAL,
                Distrito,
                CALIDAD_GEO,
                VALIDACION_TERRITORIAL,
                LATITUD,
                LONGITUD,
                USO_GEO,
                APTO_DISTANCIA_A_OFICINA
            FROM resultados
            ORDER BY FILA_ID
            """,
            conn
        )

    print("\n" + "=" * 85)
    print("RESUMEN DE CALIDAD - V3")
    print("=" * 85)
    display(resumen)

    print("\nDETALLE DE LAS DIRECCIONES")
    display(detalle)

    print("\nTOTAL PROCESADAS:", len(detalle))

else:
    print("\nLa base V3 todavía no contiene resultados.")
    print("Revisa el log mostrado arriba.")

GOOGLE MAPS CLIENTES V3 - PRUEBA CONTROLADA

1. VERIFICANDO CONFIGURACIÓN Y POLÍGONOS INEI
Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
Base preparada: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\direcciones_preparadas_google_masivo.sqlite | 159,459 filas
Ejemplo distrito: ('LIMA', 'LIMA', 'CHACLACAYO')
¿Existe polígono del ejemplo?: True
Sin consultas web y sin modificaciones de resultados.


2. INICIANDO PRUEBA DE 20 DIRECCIONES
Los resultados se guardarán en una base V3 independiente.

Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
GOOGLE MAPS CLIENTES V3 - FILTRO INEI / REFERENCIAS PROVISIONALES
Laptop       : 1
Total global : 159,459
Rango        : 1->20 ASC
Resultados   : D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\google_resultados_laptop1_v3.sqlite
Guardado     : COMMIT después de cada dirección; resultados en SQLite V3 separado
Red          : si se cae, espera y reintenta la misma fila
CAPTCHA      : se detiene; NO se eva

,ESTADO,CALIDAD_GEO,USO_GEO,DIRECCIONES
0,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,REFERENCIA_PROVISIONAL,8
1,SIN_UBICACION,SIN_UBICACION,NO_USAR,7
2,ENCONTRADO_REFERENCIA,COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR,REFERENCIA_PROVISIONAL,4
3,DIRECCION_INVALIDA,DIRECCION_INVALIDA,NO_USAR,1



DETALLE DE LAS DIRECCIONES


,FILA_ID,DIRECCION_ORIGINAL,Distrito,CALIDAD_GEO,VALIDACION_TERRITORIAL,LATITUD,LONGITUD,USO_GEO,APTO_DISTANCIA_A_OFICINA
0,1,CAL HUASCATA MZ B CHACLACAYO,CHACLACAYO,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
1,2,CAL LAS BEGONIAS MZ C URB LOS PEREGRINOS LOS O...,LOS OLIVOS,REFERENCIA_ZONA,DENTRO_DISTRITO,-11.973470,-77.073291,REFERENCIA_PROVISIONAL,0
2,3,JR PUNO URB ORCOTIUNA ORCOTUNA,ORCOTUNA,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
3,4,CAL ALMIRANTE GRAU BARRIO QUINAPATA 913 RIOJA,RIOJA,REFERENCIA_ZONA,DENTRO_DISTRITO,-6.057048,-77.165772,REFERENCIA_PROVISIONAL,0
4,5,AV BATANGRANDE 876 URB SAGRADO CORAZON DE JESU...,FERREÑAFE,COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR,DENTRO_DISTRITO,-6.625085,-79.786210,REFERENCIA_PROVISIONAL,0
5,6,JR VILCANOTA MZ 10 URB ACHO MEGO YARINACOCHA,YARINACOCHA,REFERENCIA_ZONA,DENTRO_DISTRITO,-8.388753,-74.584367,REFERENCIA_PROVISIONAL,0
6,7,MZ 7 URB HORACIO ZEBALLOS GAMEZ SOCABAYA,SOCABAYA,REFERENCIA_ZONA,DENTRO_DISTRITO,-16.489972,-71.506367,REFERENCIA_PROVISIONAL,0
7,8,AV CARRETERA CENTRAL 405 ATE,ATE,REFERENCIA_ZONA,DENTRO_DISTRITO,-12.010324,-76.871470,REFERENCIA_PROVISIONAL,0
8,9,AV CIRCUNVALACION 2135 URB TARAPOTO TARAPOTO,TARAPOTO,COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR,DENTRO_DISTRITO,-6.475129,-76.372398,REFERENCIA_PROVISIONAL,0
9,10,CAL LA LIBERTAD SECTOR CAMPO SERIAL 100 URB VA...,PICHARI,REFERENCIA_ZONA,DENTRO_DISTRITO,-12.505988,-73.821852,REFERENCIA_PROVISIONAL,0



TOTAL PROCESADAS: 20


In [13]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE = Path(
    r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones"
)

DB_V2 = BASE / "google_resultados_laptop1_v2.sqlite"
DB_V3 = BASE / "google_resultados_laptop1_v3.sqlite"

print("=" * 90)
print("PASO 7 - DIAGNOSTICO DE RECHAZOS GOOGLE MAPS V3")
print("=" * 90)

if not DB_V3.exists():
    raise FileNotFoundError(DB_V3)

# ==========================================================
# 1. LEER RESULTADOS V3
# ==========================================================

with sqlite3.connect(DB_V3) as conn:

    df_v3 = pd.read_sql_query("""
        SELECT
            FILA_ID,
            DIRECCION_ORIGINAL,
            Distrito,
            ESTADO,
            CALIDAD_GEO,
            VALIDACION_TERRITORIAL,
            LATITUD,
            LONGITUD,
            USO_GEO
        FROM resultados
        WHERE FILA_ID BETWEEN 1 AND 20
        ORDER BY FILA_ID
    """, conn)

    tablas = pd.read_sql_query("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
    """, conn)["name"].tolist()

    if "auditoria_v3" not in tablas:
        raise RuntimeError(
            "No se encontro la tabla auditoria_v3"
        )

    df_candidatos = pd.read_sql_query("""
        SELECT
            FILA_ID,
            QUERY_TIPO,
            NOMBRE_GOOGLE,
            DIRECCION_GOOGLE,
            LATITUD,
            LONGITUD,
            TERRITORIO,
            CALIDAD,
            SCORE_AUDITORIA,
            MOTIVO,
            ELEGIDO
        FROM auditoria_v3
        WHERE FILA_ID BETWEEN 1 AND 20
    """, conn)

# ==========================================================
# 2. COMPARAR V2 VS V3
# ==========================================================

if DB_V2.exists():

    with sqlite3.connect(DB_V2) as conn:
        df_v2 = pd.read_sql_query("""
            SELECT
                FILA_ID,
                ESTADO AS ESTADO_V2,
                LATITUD AS LATITUD_V2,
                LONGITUD AS LONGITUD_V2
            FROM resultados
            WHERE FILA_ID BETWEEN 1 AND 20
        """, conn)

else:
    df_v2 = pd.DataFrame(columns=[
        "FILA_ID", "ESTADO_V2",
        "LATITUD_V2", "LONGITUD_V2"
    ])

comparacion = df_v3.merge(
    df_v2,
    on="FILA_ID",
    how="left"
)

comparacion["CAMBIO"] = comparacion.apply(
    lambda r: (
        "RECUPERADA"
        if pd.notna(r["LATITUD"]) and pd.isna(r["LATITUD_V2"])
        else "DESCARTADA_V3"
        if pd.isna(r["LATITUD"]) and pd.notna(r["LATITUD_V2"])
        else "CON_COORDENADA"
        if pd.notna(r["LATITUD"])
        else "SIN_COORDENADA"
    ),
    axis=1
)

print("\n1. COMPARACION GENERAL V2 VS V3")
print(
    comparacion["CAMBIO"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n2. DIRECCIONES DESCARTADAS POR V3")

descartadas = comparacion[
    comparacion["CAMBIO"] == "DESCARTADA_V3"
]

print(
    descartadas[
        [
            "FILA_ID",
            "DIRECCION_ORIGINAL",
            "Distrito",
            "ESTADO",
            "CALIDAD_GEO"
        ]
    ].to_string(index=False)
)

# ==========================================================
# 3. MOTIVOS DE RECHAZO DE LOS CANDIDATOS
# ==========================================================

print("\n3. MOTIVOS DE RECHAZO")

rechazos = df_candidatos[
    ~df_candidatos["CALIDAD"].fillna("").isin([
        "REFERENCIA_ZONA",
        "REFERENCIA_VIA",
        "COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR",
        "COINCIDENCIA_MZ_LT_POR_VERIFICAR"
    ])
]

print(
    rechazos["MOTIVO"]
    .fillna("SIN MOTIVO")
    .value_counts()
    .head(15)
    .to_string()
)

# ==========================================================
# 4. REVISAR LOS CASOS MAS IMPORTANTES
# ==========================================================

print("\n4. CANDIDATOS PARA REVISION DETALLADA")

FILAS = [1, 3, 8, 11, 12, 13, 15, 16, 17, 19, 20]

for fila in FILAS:

    original = df_v3[df_v3["FILA_ID"] == fila]

    if original.empty:
        continue

    r = original.iloc[0]

    print("\n" + "-" * 90)
    print(f"FILA {fila}: {r['DIRECCION_ORIGINAL']}")
    print(f"DISTRITO: {r['Distrito']}")
    print(f"RESULTADO V3: {r['CALIDAD_GEO']}")

    muestra = df_candidatos[
        df_candidatos["FILA_ID"] == fila
    ].copy()

    if muestra.empty:
        print("SIN CANDIDATOS GUARDADOS")
        continue

    muestra["ES_VALIDO"] = muestra["CALIDAD"].isin([
        "REFERENCIA_ZONA",
        "REFERENCIA_VIA",
        "COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR",
        "COINCIDENCIA_MZ_LT_POR_VERIFICAR"
    ])

    muestra["DENTRO"] = (
        muestra["TERRITORIO"] == "DENTRO_DISTRITO"
    )

    muestra = muestra.sort_values(
        ["ES_VALIDO", "DENTRO", "SCORE_AUDITORIA"],
        ascending=[False, False, False]
    )

    muestra = muestra.drop_duplicates(
        subset=[
            "NOMBRE_GOOGLE",
            "LATITUD",
            "LONGITUD",
            "MOTIVO"
        ]
    ).head(4)

    for _, c in muestra.iterrows():
        print(
            f"\n  Google: {str(c['NOMBRE_GOOGLE'])[:65]}"
            f"\n  Direccion: {str(c['DIRECCION_GOOGLE'])[:90]}"
            f"\n  Territorio: {c['TERRITORIO']}"
            f"\n  Clasificacion: {c['CALIDAD']}"
            f"\n  Motivo: {c['MOTIVO']}"
            f"\n  Coordenadas: {c['LATITUD']}, {c['LONGITUD']}"
        )

# ==========================================================
# 5. EXPORTAR DIAGNOSTICO
# ==========================================================

archivo_comparacion = BASE / "diagnostico_comparacion_v2_v3.csv"
archivo_rechazos = BASE / "diagnostico_rechazos_v3.csv"

comparacion.to_csv(
    archivo_comparacion,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

df_candidatos.to_csv(
    archivo_rechazos,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 90)
print("DIAGNOSTICO FINALIZADO")
print("=" * 90)

print("\nArchivos generados:")
print(archivo_comparacion)
print(archivo_rechazos)

PASO 7 - DIAGNOSTICO DE RECHAZOS GOOGLE MAPS V3

1. COMPARACION GENERAL V2 VS V3
CAMBIO
CON_COORDENADA    12
DESCARTADA_V3      8

2. DIRECCIONES DESCARTADAS POR V3
 FILA_ID                                       DIRECCION_ORIGINAL             Distrito             ESTADO        CALIDAD_GEO
       1                             CAL HUASCATA MZ B CHACLACAYO           CHACLACAYO      SIN_UBICACION      SIN_UBICACION
       3                           JR PUNO URB ORCOTIUNA ORCOTUNA             ORCOTUNA      SIN_UBICACION      SIN_UBICACION
      11               CAL LAS BEGONIAS 750 LA BANDA DE SCHILCAYO LA BANDA DE SHILCAYO      SIN_UBICACION      SIN_UBICACION
      12 CAL SIN NOMBRE MZ 1A URB TABLADAD DE LURIN V EL SALVADOR    VILLA EL SALVADOR      SIN_UBICACION      SIN_UBICACION
      15                                  CAL ASILLO 017 SAN JOSE             SAN JOSÉ      SIN_UBICACION      SIN_UBICACION
      16                                    CAL XDEPURAR 88888888                 LIM

In [14]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
CARPETA = BASE / "clientes_direcciones"

SCRIPT = BASE / "07_google_maps_clientes_v4.py"

print("=" * 85)
print("PASO 8 - AUDITORIA DE RECUPERACION GOOGLE MAPS V4")
print("=" * 85)

# 1. Verificar archivos
if not SCRIPT.is_file():
    raise FileNotFoundError(f"Falta el robot: {SCRIPT}")

db_v3 = CARPETA / "google_resultados_laptop1_v3.sqlite"

if not db_v3.is_file():
    raise FileNotFoundError(f"Falta la base V3: {db_v3}")

# 2. Reevaluar candidatos sin consultar Google
resultado = subprocess.run(
    [
        sys.executable,
        "-u",
        str(SCRIPT),
        "--laptop", "1",
        "--auditar-v3"
    ],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False
)

print(resultado.stdout)

if resultado.stderr:
    print("\nDETALLE:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        f"Error en auditoria V4: {resultado.returncode}"
    )

# 3. Leer el resumen V4
archivo = CARPETA / "auditoria_v4_resumen_laptop1.csv"

df = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8-sig"
)

print("\n" + "=" * 85)
print("COMPARACION V3 VS PROPUESTA V4")
print("=" * 85)

print("\n1. RESUMEN DE DECISIONES")
print(
    df["DECISION_V4"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n2. CALIDAD PROPUESTA")
print(
    df["CALIDAD_PROPUESTA_V4"]
    .value_counts(dropna=False)
    .to_string()
)

# 4. Mostrar direcciones que cambiaron
cambios = df[
    df["DECISION_V4"].isin([
        "PROPUESTA_RECUPERADA_REVISAR",
        "PROPUESTA_CAMBIO_REVISAR",
        "DESCARTADA_V4_REVISAR"
    ])
].copy()

print("\n3. DIRECCIONES CON CAMBIOS")
print(f"Cantidad: {len(cambios)}")

columnas = [
    "FILA_ID",
    "DIRECCION",
    "DISTRITO",
    "CALIDAD_V3",
    "DECISION_V4",
    "CALIDAD_PROPUESTA_V4",
    "LAT_PROPUESTA_V4",
    "LON_PROPUESTA_V4",
    "NOMBRE_PROPUESTO",
    "MOTIVO_PROPUESTA"
]

pd.set_option("display.max_colwidth", 75)

if len(cambios):
    display(cambios[columnas])
else:
    print("Ninguna propuesta cambió respecto a V3.")

print("\n" + "=" * 85)
print("AUDITORIA FINALIZADA")
print("=" * 85)

PASO 8 - AUDITORIA DE RECUPERACION GOOGLE MAPS V4
Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
V4 - REEVALUACION DE CANDIDATOS V3 (SIN GOOGLE)
Direcciones evaluadas: 20 | candidatos: 118
  CONSERVA_O_MUY_CERCANA                  12
  CONTINUA_SIN_UBICACION                   6
  DIRECCION_INVALIDA                       1
  PROPUESTA_RECUPERADA_REVISAR             1
Resumen: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\auditoria_v4_resumen_laptop1.csv
Candidatos: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\auditoria_v4_candidatos_laptop1.csv
No se modificó ninguna SQLite; las propuestas NO son predios certificados.


COMPARACION V3 VS PROPUESTA V4

1. RESUMEN DE DECISIONES
DECISION_V4
CONSERVA_O_MUY_CERCANA          12
CONTINUA_SIN_UBICACION           6
PROPUESTA_RECUPERADA_REVISAR     1
DIRECCION_INVALIDA               1

2. CALIDAD PROPUESTA
CALIDAD_PROPUESTA_V4
REFERENCIA_ZONA                          8
SIN_UBICACION                            7

,FILA_ID,DIRECCION,DISTRITO,CALIDAD_V3,DECISION_V4,CALIDAD_PROPUESTA_V4,LAT_PROPUESTA_V4,LON_PROPUESTA_V4,NOMBRE_PROPUESTO,MOTIVO_PROPUESTA
0,1,CAL HUASCATA MZ B CHACLACAYO,CHACLACAYO,SIN_UBICACION,PROPUESTA_RECUPERADA_REVISAR,REFERENCIA_VIA_POR_VERIFICAR,-11.994135,-76.821341,Huascata Mz B,"Nombre completo igual a la vía, sin prefijo CAL/JR/AV; dentro del distr..."



AUDITORIA FINALIZADA


In [15]:
from pathlib import Path
import subprocess
import sys
import sqlite3
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
CARPETA = BASE / "clientes_direcciones"

SCRIPT = BASE / "07_google_maps_clientes_v4.py"
DB_PREP = CARPETA / "direcciones_preparadas_google_masivo.sqlite"
DB_V4 = CARPETA / "google_resultados_laptop1_v4.sqlite"

DESDE = 21
HASTA = 30

print("=" * 85)
print("PRUEBA REAL GOOGLE MAPS V4 - 10 DIRECCIONES NUEVAS")
print("=" * 85)

# ======================================================
# 1. VERIFICAR ARCHIVOS
# ======================================================

for nombre, ruta in [
    ("Robot V4", SCRIPT),
    ("Base preparada", DB_PREP)
]:
    print(f"{nombre}: {'OK' if ruta.exists() else 'FALTA'}")

    if not ruta.is_file():
        raise FileNotFoundError(ruta)

# ======================================================
# 2. VERIFICAR CONFIGURACION
# ======================================================

print("\nVERIFICANDO CONFIGURACION...")

verificacion = subprocess.run(
    [
        sys.executable,
        "-u",
        str(SCRIPT),
        "--laptop", "1",
        "--verificar"
    ],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(verificacion.stdout)

if verificacion.returncode != 0:
    print(verificacion.stderr)
    raise RuntimeError("Error de configuracion")

# ======================================================
# 3. EJECUTAR BUSQUEDAS REALES
# ======================================================

print("\n" + "=" * 85)
print(f"BUSCANDO FILAS {DESDE} A {HASTA}")
print("=" * 85)

comando = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--laptop", "1",
    "--desde", str(DESDE),
    "--hasta", str(HASTA),
    "--max-filas", "10"
]

proceso = subprocess.Popen(
    comando,
    cwd=str(BASE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

try:
    for linea in proceso.stdout:
        print(linea, end="", flush=True)

    codigo = proceso.wait()

except KeyboardInterrupt:
    proceso.terminate()
    proceso.wait()
    print("\nEjecucion interrumpida.")
    print("Las filas ya guardadas se conservaran.")
    raise

print("\nCODIGO DE SALIDA:", codigo)

# ======================================================
# 4. LEER RESULTADOS
# ======================================================

if not DB_V4.is_file():
    raise RuntimeError(
        "No se genero la base SQLite V4. "
        "Revisa los mensajes anteriores."
    )

with sqlite3.connect(DB_V4) as conn:

    resumen = pd.read_sql_query(
        """
        SELECT
            ESTADO,
            CALIDAD_GEO,
            USO_GEO,
            COUNT(*) AS DIRECCIONES
        FROM resultados
        WHERE FILA_ID BETWEEN ? AND ?
        GROUP BY ESTADO, CALIDAD_GEO, USO_GEO
        ORDER BY DIRECCIONES DESC
        """,
        conn,
        params=(DESDE, HASTA)
    )

    detalle = pd.read_sql_query(
        """
        SELECT
            FILA_ID,
            DIRECCION_ORIGINAL,
            Distrito,
            ESTADO,
            CALIDAD_GEO,
            VALIDACION_TERRITORIAL,
            LATITUD,
            LONGITUD,
            USO_GEO,
            APTO_DISTANCIA_A_OFICINA
        FROM resultados
        WHERE FILA_ID BETWEEN ? AND ?
        ORDER BY FILA_ID
        """,
        conn,
        params=(DESDE, HASTA)
    )

# ======================================================
# 5. RESUMEN EJECUTIVO
# ======================================================

total = len(detalle)

referencias = int(
    (detalle["USO_GEO"] == "REFERENCIA_PROVISIONAL").sum()
)

sin_ubicacion = total - referencias

print("\n" + "=" * 85)
print("RESUMEN DE LA PRUEBA V4")
print("=" * 85)

print(f"Direcciones procesadas: {total:,}")
print(f"Referencias provisionales: {referencias:,}")
print(f"Sin referencia utilizable: {sin_ubicacion:,}")

if total > 0:
    print(f"Cobertura provisional: {referencias / total:.1%}")

print("\nCLASIFICACION")
display(resumen)

print("\nDETALLE")
display(detalle)

if total < HASTA - DESDE + 1:
    print("\nATENCION: la prueba no completo todas las filas.")
    print("Puedes reanudar ejecutando nuevamente este bloque.")

print("\n" + "=" * 85)
print("FIN DE LA PRUEBA")
print("=" * 85)

PRUEBA REAL GOOGLE MAPS V4 - 10 DIRECCIONES NUEVAS
Robot V4: OK
Base preparada: OK

VERIFICANDO CONFIGURACION...
Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
Base preparada: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\direcciones_preparadas_google_masivo.sqlite | 159,459 filas
Ejemplo distrito: ('LIMA', 'LIMA', 'CHACLACAYO')
¿Existe polígono del ejemplo?: True
Sin consultas web y sin modificaciones de resultados.


BUSCANDO FILAS 21 A 30
Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
GOOGLE MAPS CLIENTES V4 - FILTRO INEI / REFERENCIAS PROVISIONALES
Laptop       : 1
Total global : 159,459
Rango        : 21->30 ASC
Resultados   : D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\google_resultados_laptop1_v4.sqlite
Guardado     : COMMIT después de cada dirección; resultados en SQLite V4 separado
Red          : si se cae, espera y reintenta la misma fila
CAPTCHA      : se detiene; NO se evade
PRECISIÓN    : NO se certifican predios ni se as

,ESTADO,CALIDAD_GEO,USO_GEO,DIRECCIONES
0,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,REFERENCIA_PROVISIONAL,5
1,SIN_UBICACION,SIN_UBICACION,NO_USAR,4
2,ENCONTRADO_REFERENCIA,REFERENCIA_VIA_POR_VERIFICAR,REFERENCIA_PROVISIONAL,1



DETALLE


,FILA_ID,DIRECCION_ORIGINAL,Distrito,ESTADO,CALIDAD_GEO,VALIDACION_TERRITORIAL,LATITUD,LONGITUD,USO_GEO,APTO_DISTANCIA_A_OFICINA
0,21,JR ANDAHUAYLAS 16 MZ 1 URB HUAROCHIRI V EL SALVADOR,VILLA EL SALVADOR,SIN_UBICACION,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
1,22,CAL 11 DE JUNIO S14A LURIGANCHO CHOSICA,LURIGANCHO,SIN_UBICACION,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
2,23,CAL JOSE CARLOS MAREATEGUI 21 PUCALA,PUCALÁ,SIN_UBICACION,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
3,24,MZ Q URB CHANU CHANU PUNO,PUNO,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,DENTRO_DISTRITO,-15.857010,-70.012805,REFERENCIA_PROVISIONAL,0
4,25,CAL CASERIO CHEPITO OLIVOS 01 MORROPE,MORROPE,SIN_UBICACION,SIN_UBICACION,SIN_COORDENADA_UTIL,NaN,NaN,NO_USAR,0
5,26,CAL EMILIANO NINO 476 MOTUPE,MOTUPE,ENCONTRADO_REFERENCIA,REFERENCIA_VIA_POR_VERIFICAR,DENTRO_DISTRITO,-6.154459,-79.715992,REFERENCIA_PROVISIONAL,0
6,27,CAL SIN NOMBRE MZ K URB LOS NISPEROS SAN M DE PORRES,SAN MARTIN DE PORRES,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,DENTRO_DISTRITO,-11.998449,-77.098332,REFERENCIA_PROVISIONAL,0
7,28,CAL MZ B URB SAN LUIS TRUJILLO,TRUJILLO,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,DENTRO_DISTRITO,-8.096815,-79.043932,REFERENCIA_PROVISIONAL,0
8,29,PSJ CESAR VALLEJO MZ 3 URB EL TRIUNFO LA ESPERANZA,LA ESPERANZA,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,DENTRO_DISTRITO,-8.049287,-79.054963,REFERENCIA_PROVISIONAL,0
9,30,CAL ORTEGA Y GAZET 498 URB LA NORIA TRUJILLO,TRUJILLO,ENCONTRADO_REFERENCIA,REFERENCIA_ZONA,DENTRO_DISTRITO,-8.103773,-79.010331,REFERENCIA_PROVISIONAL,0



FIN DE LA PRUEBA


In [16]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE = Path(
    r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones"
)

DB_V4 = BASE / "google_resultados_laptop1_v4.sqlite"
DB_PREP = BASE / "direcciones_preparadas_google_masivo.sqlite"

DESDE = 21
HASTA = 30

print("=" * 90)
print("PASO 9 - DIAGNOSTICO DE CALIDAD GOOGLE MAPS V4")
print("=" * 90)

for archivo in [DB_V4, DB_PREP]:
    if not archivo.is_file():
        raise FileNotFoundError(archivo)

# ======================================================
# 1. LEER DATOS EN MODO SOLO LECTURA
# ======================================================

with sqlite3.connect(
    DB_V4.as_uri() + "?mode=ro",
    uri=True
) as conn:

    resultados = pd.read_sql_query(
        """
        SELECT
            FILA_ID,
            DIRECCION_ORIGINAL,
            Distrito,
            ESTADO,
            CALIDAD_GEO,
            LATITUD,
            LONGITUD,
            USO_GEO
        FROM resultados
        WHERE FILA_ID BETWEEN ? AND ?
        ORDER BY FILA_ID
        """,
        conn,
        params=(DESDE, HASTA)
    )

    auditoria = pd.read_sql_query(
        """
        SELECT
            FILA_ID,
            QUERY_TIPO,
            QUERY,
            NOMBRE_GOOGLE,
            DIRECCION_GOOGLE,
            LATITUD,
            LONGITUD,
            TERRITORIO,
            CALIDAD,
            SCORE_AUDITORIA,
            MOTIVO,
            ELEGIDO
        FROM auditoria_v4
        WHERE FILA_ID BETWEEN ? AND ?
        """,
        conn,
        params=(DESDE, HASTA)
    )

with sqlite3.connect(
    DB_PREP.as_uri() + "?mode=ro",
    uri=True
) as conn:

    columnas = [
        "FILA_ID", "Direccion",
        "NUMERO_BUSCADO", "MANZANA", "LOTE",
        "VIA_CORREGIDA", "ZONA_CORREGIDA"
    ]

    columnas += [
        f"Q{i}" for i in range(1, 13)
    ]

    preparadas = pd.read_sql_query(
        f"""
        SELECT {", ".join(columnas)}
        FROM preparadas
        WHERE FILA_ID BETWEEN ? AND ?
        ORDER BY FILA_ID
        """,
        conn,
        params=(DESDE, HASTA)
    )

# ======================================================
# 2. RESUMEN DE LOS CANDIDATOS
# ======================================================

print("\n1. CANDIDATOS GENERADOS")

print(f"Direcciones procesadas: {len(resultados)}")
print(f"Candidatos evaluados: {len(auditoria)}")

print("\nESTADO TERRITORIAL")
print(
    auditoria["TERRITORIO"]
    .fillna("SIN INFORMACION")
    .value_counts()
    .to_string()
)

print("\nMOTIVOS DE RECHAZO")
rechazos = auditoria[
    auditoria["ELEGIDO"].fillna(0).eq(0)
]

print(
    rechazos["MOTIVO"]
    .fillna("SIN MOTIVO")
    .value_counts()
    .head(12)
    .to_string()
)

# ======================================================
# 3. REVISION POR DIRECCION
# ======================================================

print("\n" + "=" * 90)
print("2. DIAGNOSTICO INDIVIDUAL")
print("=" * 90)

for _, r in resultados.iterrows():

    fila = int(r["FILA_ID"])

    print("\n" + "-" * 90)
    print(f"FILA {fila}: {r['DIRECCION_ORIGINAL']}")
    print(f"DISTRITO: {r['Distrito']}")
    print(f"RESULTADO: {r['CALIDAD_GEO']}")

    prep = preparadas[
        preparadas["FILA_ID"] == fila
    ]

    if not prep.empty:

        p = prep.iloc[0]

        print(
            f"VIA EXTRAIDA: {p['VIA_CORREGIDA']}"
        )
        print(
            f"NUMERO EXTRAIDO: {p['NUMERO_BUSCADO']}"
        )
        print(
            f"MANZANA: {p['MANZANA']} | "
            f"LOTE: {p['LOTE']}"
        )
        print(
            f"ZONA: {p['ZONA_CORREGIDA']}"
        )

        if r["USO_GEO"] != "REFERENCIA_PROVISIONAL":

            print("\nPRIMERAS CONSULTAS GENERADAS:")

            for i in range(1, 7):
                valor = p.get(f"Q{i}", "")

                if pd.notna(valor) and str(valor).strip():
                    print(f" Q{i}: {valor}")

    candidatos = auditoria[
        auditoria["FILA_ID"] == fila
    ].copy()

    if candidatos.empty:
        print("\nGoogle no dejo candidatos auditables.")
        continue

    dentro = candidatos[
        candidatos["TERRITORIO"] == "DENTRO_DISTRITO"
    ]

    print(
        f"\nCandidatos: {len(candidatos)} | "
        f"Dentro del distrito: {len(dentro)}"
    )

    candidatos["ORDEN_ELEGIDO"] = (
        candidatos["ELEGIDO"]
        .fillna(0)
        .astype(int)
    )

    candidatos["ORDEN_TERRITORIO"] = (
        candidatos["TERRITORIO"]
        .eq("DENTRO_DISTRITO")
        .astype(int)
    )

    candidatos = candidatos.sort_values(
        [
            "ORDEN_ELEGIDO",
            "ORDEN_TERRITORIO",
            "SCORE_AUDITORIA"
        ],
        ascending=[False, False, False]
    )

    candidatos = candidatos.drop_duplicates(
        subset=[
            "NOMBRE_GOOGLE",
            "LATITUD",
            "LONGITUD",
            "MOTIVO"
        ]
    )

    limite = (
        5 if r["USO_GEO"] != "REFERENCIA_PROVISIONAL"
        else 2
    )

    print("\nMEJORES CANDIDATOS Y MOTIVOS:")

    for _, c in candidatos.head(limite).iterrows():

        print(
            f"\n Nombre: {c['NOMBRE_GOOGLE']}"
            f"\n Direccion: {c['DIRECCION_GOOGLE']}"
            f"\n Territorio: {c['TERRITORIO']}"
            f"\n Calidad: {c['CALIDAD']}"
            f"\n Motivo: {c['MOTIVO']}"
            f"\n Coordenadas: "
            f"{c['LATITUD']}, {c['LONGITUD']}"
        )

# ======================================================
# 4. GUARDAR DIAGNOSTICO
# ======================================================

salida = BASE / "diagnostico_calidad_v4_filas_21_30.csv"

auditoria.to_csv(
    salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 90)
print("DIAGNOSTICO FINALIZADO")
print("=" * 90)

print("Archivo generado:")
print(salida)

PASO 9 - DIAGNOSTICO DE CALIDAD GOOGLE MAPS V4

1. CANDIDATOS GENERADOS
Direcciones procesadas: 10
Candidatos evaluados: 74

ESTADO TERRITORIAL
TERRITORIO
DENTRO_DISTRITO    60
FUERA_DISTRITO     13
SIN_COORDENADAS     1

MOTIVOS DE RECHAZO
MOTIVO
Coordenada en distrito pero sin evidencia suficiente de dirección                                       31
Dentro del distrito; evidencia de texto, NO predio certificado                                          18
Candidato no validado dentro del polígono: FUERA_DISTRITO                                               13
Nombre completo igual a la vía, sin prefijo CAL/JR/AV; dentro del distrito INEI; NO es predio exacto     2
Se busca Mz 1, Google dice Mz J                                                                          1
Se busca Mz 1, Google dice Mz 9                                                                          1
Se busca Mz 1, Google dice Mz N                                                                          1
Can

In [18]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
SCRIPT = BASE / "08_google_maps_clientes_v5.py"
CARPETA = BASE / "clientes_direcciones"

print("=" * 85)
print("PASO 10 - AUDITORIA V5 SOBRE RESULTADOS V4")
print("=" * 85)

if not SCRIPT.is_file():
    raise FileNotFoundError(f"No existe: {SCRIPT}")

comando = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--laptop", "1",
    "--auditar-v4",
    "--desde", "21",
    "--hasta", "30"
]

resultado = subprocess.run(
    comando,
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False
)

print(resultado.stdout)

if resultado.stderr:
    print("\nDETALLE DEL ERROR:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        f"Auditoria V5 fallida: {resultado.returncode}"
    )

ARCHIVO = CARPETA / "auditoria_v5_resumen_laptop1.csv"

df = pd.read_csv(
    ARCHIVO,
    sep=";",
    encoding="utf-8-sig"
)

print("\n" + "=" * 85)
print("COMPARACION V4 VS V5")
print("=" * 85)

print("\nDECISIONES:")
print(
    df["DECISION_V5"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nCALIDAD PROPUESTA:")
print(
    df["CALIDAD_PROPUESTA_V5"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nDETALLE POR DIRECCION:")

columnas = [
    "FILA_ID",
    "DIRECCION",
    "CALIDAD_V4",
    "DECISION_V5",
    "CALIDAD_PROPUESTA_V5",
    "NOMBRE_PROPUESTO",
    "USO_GEO_PROPUESTO",
    "LATITUD_PROPUESTA",
    "LONGITUD_PROPUESTA"
]

pd.set_option("display.max_colwidth", 65)

display(df[columnas].sort_values("FILA_ID"))

print("\nArchivo generado:", ARCHIVO)
print("\nAUDITORIA FINALIZADA")

PASO 10 - AUDITORIA V5 SOBRE RESULTADOS V4
Capa INEI: DISTRITO.gpkg | 1,890 polígonos | 1,890 divisiones
V5 - AUDITORIA SOBRE CANDIDATOS V4 - SIN GOOGLE
Direcciones: 10 | Candidatos: 74
CONSERVA_O_MUY_CERCANA: 5
CONTINUA_SIN_UBICACION: 2
PROPUESTA_CAMBIO_REVISAR: 1
RECUPERADA_REVISAR: 2
FILA    21 |     CONTINUA_SIN_UBICACION | SIN_UBICACION                                | 
FILA    22 |         RECUPERADA_REVISAR | REFERENCIA_VIA_POR_VERIFICAR                 | 11 De Junio
FILA    23 |         RECUPERADA_REVISAR | REFERENCIA_VIA_GRAFIA_REVISAR                | José Carlos Mariátegui
FILA    24 |     CONSERVA_O_MUY_CERCANA | REFERENCIA_ZONA                              | Urbanización Chanu Chanu Primera Etapa
FILA    25 |     CONTINUA_SIN_UBICACION | SIN_UBICACION                                | 
FILA    26 |     CONSERVA_O_MUY_CERCANA | REFERENCIA_VIA_POR_VERIFICAR                 | Emiliano Niño 476
FILA    27 |     CONSERVA_O_MUY_CERCANA | REFERENCIA_ZONA                           

,FILA_ID,DIRECCION,CALIDAD_V4,DECISION_V5,CALIDAD_PROPUESTA_V5,NOMBRE_PROPUESTO,USO_GEO_PROPUESTO,LATITUD_PROPUESTA,LONGITUD_PROPUESTA
0,21,JR ANDAHUAYLAS 16 MZ 1 URB HUAROCHIRI V EL SALVADOR,SIN_UBICACION,CONTINUA_SIN_UBICACION,SIN_UBICACION,NaN,NO_USAR,NaN,NaN
1,22,CAL 11 DE JUNIO S14A LURIGANCHO CHOSICA,SIN_UBICACION,RECUPERADA_REVISAR,REFERENCIA_VIA_POR_VERIFICAR,11 De Junio,REFERENCIA_PROVISIONAL,-11.973546,-76.938581
2,23,CAL JOSE CARLOS MAREATEGUI 21 PUCALA,SIN_UBICACION,RECUPERADA_REVISAR,REFERENCIA_VIA_GRAFIA_REVISAR,José Carlos Mariátegui,REFERENCIA_PROVISIONAL,-6.780829,-79.609504
3,24,MZ Q URB CHANU CHANU PUNO,REFERENCIA_ZONA,CONSERVA_O_MUY_CERCANA,REFERENCIA_ZONA,Urbanización Chanu Chanu Primera Etapa,REFERENCIA_PROVISIONAL,-15.857010,-70.012805
4,25,CAL CASERIO CHEPITO OLIVOS 01 MORROPE,SIN_UBICACION,CONTINUA_SIN_UBICACION,SIN_UBICACION,NaN,NO_USAR,NaN,NaN
5,26,CAL EMILIANO NINO 476 MOTUPE,REFERENCIA_VIA_POR_VERIFICAR,CONSERVA_O_MUY_CERCANA,REFERENCIA_VIA_POR_VERIFICAR,Emiliano Niño 476,REFERENCIA_PROVISIONAL,-6.154459,-79.715992
6,27,CAL SIN NOMBRE MZ K URB LOS NISPEROS SAN M DE PORRES,REFERENCIA_ZONA,CONSERVA_O_MUY_CERCANA,REFERENCIA_ZONA,Parque Principal Los Nísperos - SMP,REFERENCIA_PROVISIONAL,-11.998449,-77.098332
7,28,CAL MZ B URB SAN LUIS TRUJILLO,REFERENCIA_ZONA,CONSERVA_O_MUY_CERCANA,REFERENCIA_ZONA,Fitness Place,REFERENCIA_PROVISIONAL,-8.096815,-79.043932
8,29,PSJ CESAR VALLEJO MZ 3 URB EL TRIUNFO LA ESPERANZA,REFERENCIA_ZONA,CONSERVA_O_MUY_CERCANA,REFERENCIA_ZONA,Parque triunfo los 4 suyos,REFERENCIA_PROVISIONAL,-8.049287,-79.054963
9,30,CAL ORTEGA Y GAZET 498 URB LA NORIA TRUJILLO,REFERENCIA_ZONA,PROPUESTA_CAMBIO_REVISAR,VIA_NUMERO_TIPO_DISTINTO_REVISAR,Pje. Ortega y Gasset 498,REVISION_MANUAL,-8.101817,-79.007377



Archivo generado: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\auditoria_v5_resumen_laptop1.csv

AUDITORIA FINALIZADA


In [20]:
from pathlib import Path
import subprocess
import sys

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")

SCRIPT = BASE / "09_catalogo_respaldo_distrital_clientes.py"

CSV = BASE.parent / "direcciones.csv"

GPKG = (
    BASE / "clientes_direcciones"
    / "limites_inei"
    / "DISTRITO.gpkg"
)

print("=" * 85)
print("PASO 11 - GEOLOCALIZACION DISTRITAL DE CLIENTES")
print("=" * 85)

# 1. Verificar archivos
for nombre, archivo in [
    ("Script", SCRIPT),
    ("Direcciones CSV", CSV),
    ("Limites INEI", GPKG)
]:
    print(f"{nombre}: {'OK' if archivo.is_file() else 'FALTA'}")

    if not archivo.is_file():
        raise FileNotFoundError(archivo)

# 2. Ejecutar el proceso
print("\nGenerando catalogo y coordenadas distritales...\n")

proceso = subprocess.Popen(
    [
        sys.executable,
        "-u",
        str(SCRIPT)
    ],
    cwd=str(BASE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

for linea in proceso.stdout:
    print(linea, end="", flush=True)

codigo = proceso.wait()

print("\nCODIGO DE SALIDA:", codigo)

if codigo != 0:
    raise RuntimeError(
        "El proceso no finalizo correctamente. "
        "Revisa el error mostrado arriba."
    )

print("\nPASO 11 FINALIZADO")

PASO 11 - GEOLOCALIZACION DISTRITAL DE CLIENTES
Script: OK
Direcciones CSV: OK
Limites INEI: OK

Generando catalogo y coordenadas distritales...

PASO 11 - CATÁLOGO INEI Y RESPALDO GEO PARA TODAS LAS DIRECCIONES
Capa geográfica: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\limites_inei\DISTRITO.gpkg
Polígonos INEI leídos: 1,890 | CRS: EPSG:4326
Distritos con punto interior: 1,890
Bloque 1: 20,000 registros procesados | 19,186 con respaldo distrital
Bloque 2: 40,000 registros procesados | 38,346 con respaldo distrital
Bloque 3: 60,000 registros procesados | 57,468 con respaldo distrital
Bloque 4: 80,000 registros procesados | 76,625 con respaldo distrital
Bloque 5: 100,000 registros procesados | 95,840 con respaldo distrital
Bloque 6: 120,000 registros procesados | 114,954 con respaldo distrital
Bloque 7: 140,000 registros procesados | 134,112 con respaldo distrital
Bloque 8: 160,000 registros procesados | 153,289 con respaldo distrital
Bloque 9: 180,000 registros procesados 

In [21]:
from pathlib import Path
import pandas as pd

BASE = Path(
    r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones"
)

ARCHIVO_BASE = BASE / "clientes_base_geografica_provisional.csv"
ARCHIVO_CATALOGO = BASE / "catalogo_centros_representativos_inei.csv"
ARCHIVO_PENDIENTES = BASE / "diagnostico_distritos_sin_coincidencia.csv"

print("=" * 90)
print("PASO 12 - DIAGNOSTICO DE COBERTURA GEO ESPACIAL")
print("=" * 90)

# ====================================================
# 1. VERIFICAR ARCHIVOS
# ====================================================

for archivo in [
    ARCHIVO_BASE,
    ARCHIVO_CATALOGO,
    ARCHIVO_PENDIENTES
]:
    if not archivo.is_file():
        raise FileNotFoundError(archivo)

# ====================================================
# 2. LEER BASE
# ====================================================

df = pd.read_csv(
    ARCHIVO_BASE,
    sep=";",
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False
)

catalogo = pd.read_csv(
    ARCHIVO_CATALOGO,
    sep=";",
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False
)

pendientes = pd.read_csv(
    ARCHIVO_PENDIENTES,
    sep=";",
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False
)

# ====================================================
# 3. INDICADORES GENERALES
# ====================================================

total = len(df)

con_geo = (
    df["LATITUD_FINAL"].str.strip().ne("") &
    df["LONGITUD_FINAL"].str.strip().ne("")
)

con_direccion = (
    df["Direccion Completa"].str.strip().ne("")
)

print("\n1. RESUMEN GENERAL")
print("-" * 60)

print(f"Total registros:              {total:,}")
print(f"Con coordenadas:             {con_geo.sum():,}")
print(f"Sin coordenadas:             {(~con_geo).sum():,}")
print(f"Cobertura geografica:        {con_geo.mean():.2%}")

print(f"\nCon direccion informada:     {con_direccion.sum():,}")
print(f"Sin direccion informada:     {(~con_direccion).sum():,}")

print(f"\nCon direccion y coordenada:  {(con_direccion & con_geo).sum():,}")
print(f"Sin direccion, con geo:      {((~con_direccion) & con_geo).sum():,}")

print(f"\nDistritos en catalogo:       {len(catalogo):,}")

# ====================================================
# 4. ESTADOS GEOGRAFICOS
# ====================================================

print("\n2. ESTADOS DE GEOLOCALIZACION")
print("-" * 60)

resumen_estados = (
    df.groupby("ESTADO_GEO", dropna=False)
    .size()
    .reset_index(name="REGISTROS")
    .sort_values("REGISTROS", ascending=False)
)

resumen_estados["PORCENTAJE"] = (
    resumen_estados["REGISTROS"] / total * 100
).round(2)

display(resumen_estados)

# ====================================================
# 5. PRINCIPALES DISTRITOS SIN MATCH
# ====================================================

print("\n3. DISTRITOS SIN COINCIDENCIA INEI")
print("-" * 60)

if len(pendientes):

    pendientes["REGISTROS"] = pd.to_numeric(
        pendientes["REGISTROS"],
        errors="coerce"
    ).fillna(0).astype(int)

    pendientes = pendientes.sort_values(
        "REGISTROS",
        ascending=False
    )

    print(
        f"Combinaciones administrativas sin match: "
        f"{len(pendientes):,}"
    )

    print(
        f"Registros afectados: "
        f"{pendientes['REGISTROS'].sum():,}"
    )

    display(pendientes.head(30))

else:
    print("No existen distritos pendientes de coincidencia.")

# ====================================================
# 6. REGISTROS SIN INFORMACION ADMINISTRATIVA
# ====================================================

print("\n4. DATOS ADMINISTRATIVOS FALTANTES")
print("-" * 60)

columnas_admin = [
    "adr_department_name",
    "adr_province_name",
    "adr_district_name"
]

for columna in columnas_admin:
    faltantes = df[columna].str.strip().eq("").sum()
    print(f"{columna}: {faltantes:,}")

sin_admin_completo = df[columnas_admin].apply(
    lambda columna: columna.str.strip().ne("")
).all(axis=1)

print(
    f"\nRegistros sin los tres datos administrativos: "
    f"{(~sin_admin_completo).sum():,}"
)

# ====================================================
# 7. CONTROL FINAL
# ====================================================

print("\n5. CONTROL DE CALIDAD")
print("-" * 60)

if total != 284345:
    print("ADVERTENCIA: revisar total de registros.")
else:
    print("Total de registros originales: OK")

if con_geo.any():
    incorrectos = (
        df.loc[con_geo, "APTO_DISTANCIA_A_OFICINA"]
        .astype(str)
        .str.strip()
        .ne("0")
    ).sum()

    print(
        "Coordenadas distritales marcadas incorrectamente "
        f"como aptas para distancia: {incorrectos:,}"
    )

print("\n" + "=" * 90)
print("DIAGNOSTICO FINALIZADO")
print("=" * 90)

PASO 12 - DIAGNOSTICO DE COBERTURA GEO ESPACIAL

1. RESUMEN GENERAL
------------------------------------------------------------
Total registros:              284,345
Con coordenadas:             272,482
Sin coordenadas:             11,863
Cobertura geografica:        95.83%

Con direccion informada:     159,459
Sin direccion informada:     124,886

Con direccion y coordenada:  155,424
Sin direccion, con geo:      117,058

Distritos en catalogo:       1,890

2. ESTADOS DE GEOLOCALIZACION
------------------------------------------------------------


,ESTADO_GEO,REGISTROS,PORCENTAJE
0,DISTRITO_INEI_ENCONTRADO,272482,95.83
2,SIN_COINCIDENCIA_INEI,8854,3.11
1,FALTAN_DATOS_ADMINISTRATIVOS,3009,1.06



3. DISTRITOS SIN COINCIDENCIA INEI
------------------------------------------------------------
Combinaciones administrativas sin match: 107
Registros afectados: 8,854


,DEP_KEY,PROV_KEY,DIST_KEY,REGISTROS,MOTIVO
0,TACNA,TACNA,CORONEL GREGORIO ALBARRACIN L,1613,SIN_COINCIDENCIA_INEI
1,MOQUEGUA,MSCAL NIETO,MOQUEGUA,1303,SIN_COINCIDENCIA_INEI
2,ICA,NAZCA,NAZCA,482,SIN_COINCIDENCIA_INEI
3,JUNIN,HUANCAYO,SAPALLANCA,446,SIN_COINCIDENCIA_INEI
4,PUNO,MELGAR,AYAVARI,365,SIN_COINCIDENCIA_INEI
5,ICA,NAZCA,VISTA ALEGRE,324,SIN_COINCIDENCIA_INEI
6,CUSCO,QUISPICANCHIS,OROPESA,258,SIN_COINCIDENCIA_INEI
7,AREQUIPA,CARAVELLI,CHALA,248,SIN_COINCIDENCIA_INEI
8,CUSCO,QUISPICANCHIS,OCONGATE,242,SIN_COINCIDENCIA_INEI
9,CUSCO,QUISPICANCHIS,URCOS,231,SIN_COINCIDENCIA_INEI



4. DATOS ADMINISTRATIVOS FALTANTES
------------------------------------------------------------
adr_department_name: 3,009
adr_province_name: 3,009
adr_district_name: 3,009

Registros sin los tres datos administrativos: 3,009

5. CONTROL DE CALIDAD
------------------------------------------------------------
Total de registros originales: OK
Coordenadas distritales marcadas incorrectamente como aptas para distancia: 0

DIAGNOSTICO FINALIZADO


In [22]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
SCRIPT = BASE / "10_recuperar_coincidencias_inei_clientes.py"
CARPETA = BASE / "clientes_direcciones"

print("=" * 90)
print("PASO 13 - RECUPERACION DE DISTRITOS SIN COINCIDENCIA INEI")
print("=" * 90)

if not SCRIPT.is_file():
    raise FileNotFoundError(f"No existe el script: {SCRIPT}")

resultado = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(resultado.stdout)

if resultado.stderr:
    print("\nERRORES:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        f"El proceso fallo: {resultado.returncode}"
    )

# Revisar equivalencias
archivo = CARPETA / "diagnostico_equivalencias_inei.csv"

df = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8-sig"
)

print("\nRESUMEN DE EQUIVALENCIAS")

display(
    df.groupby("RESULTADO", dropna=False)["REGISTROS"]
    .sum()
    .reset_index()
)

print("\nPRINCIPALES DISTRITOS RECUPERADOS")

recuperados = df[
    df["RESULTADO"] == "RECUPERADO_EQUIVALENCIA_VERIFICADA"
]

display(
    recuperados.sort_values(
        "REGISTROS",
        ascending=False
    ).head(25)
)

print("\nPRINCIPALES PENDIENTES")

pendientes = df[
    df["RESULTADO"] == "PENDIENTE_REVISION_MANUAL"
]

display(
    pendientes.sort_values(
        "REGISTROS",
        ascending=False
    ).head(20)
)

print("\nPASO 13 FINALIZADO")

PASO 13 - RECUPERACION DE DISTRITOS SIN COINCIDENCIA INEI
PASO 13 - CORRECCION SEGURA DE EQUIVALENCIAS DISTRITALES INEI
Registros originales : 284,345
Claves INEI únicas  : 1,890
Sin coordenadas inicialmente: 11,863
Pendientes por nombres INEI : 8,854 (107 combinaciones)

------------------------------------------------------------------------------------------
ANTES - Con coordenadas : 272,482 (95.83%)
RECUPERADOS POR EQUIVALENCIA VERIFICADA : 8,435
AHORA - Con coordenadas : 280,917 (98.79%)
SIN COORDENADAS : 3,428
  - Sin información administrativa: 3,009
  - Pendientes sin equivalencia verificada: 419

TOP 15 EQUIVALENCIAS Y SITUACIONES:
DEP_ORIGEN   PROV_ORIGEN                    DIST_ORIGEN  REGISTROS                          RESULTADO PROV_INEI_PROPUESTO                  DIST_INEI_PROPUESTO
     TACNA         TACNA CORONEL GREGORIO ALBARRACIN L.       1613 RECUPERADO_EQUIVALENCIA_VERIFICADA               TACNA CORONEL GREGORIO ALBARRACIN LANCHIPA
  MOQUEGUA   MSCAL NIETO         

,RESULTADO,REGISTROS
0,PENDIENTE_REVISION_MANUAL,419
1,RECUPERADO_EQUIVALENCIA_VERIFICADA,8435



PRINCIPALES DISTRITOS RECUPERADOS


,DEP_ORIGEN,PROV_ORIGEN,DIST_ORIGEN,REGISTROS,DEP_INEI_PROPUESTO,PROV_INEI_PROPUESTO,DIST_INEI_PROPUESTO,RESULTADO,EQUIVALENCIA,SUGERENCIA_1,SIMILITUD_1,SUGERENCIA_2,SIMILITUD_2
0,TACNA,TACNA,CORONEL GREGORIO ALBARRACIN L.,1613,TACNA,TACNA,CORONEL GREGORIO ALBARRACIN LANCHIPA,RECUPERADO_EQUIVALENCIA_VERIFICADA,TACNA / TACNA / CORONEL GREGORIO ALBARRACIN L => TACNA / TACN...,NaN,NaN,NaN,NaN
1,MOQUEGUA,MSCAL NIETO,MOQUEGUA,1303,MOQUEGUA,MARISCAL NIETO,MOQUEGUA,RECUPERADO_EQUIVALENCIA_VERIFICADA,MOQUEGUA / MSCAL NIETO / MOQUEGUA => MOQUEGUA / MARISCAL NIET...,NaN,NaN,NaN,NaN
2,ICA,NAZCA,NAZCA,482,ICA,NASCA,NASCA,RECUPERADO_EQUIVALENCIA_VERIFICADA,ICA / NAZCA / NAZCA => ICA / NASCA / NASCA,NaN,NaN,NaN,NaN
3,JUNÍN,HUANCAYO,SAPALLANCA,446,JUNIN,HUANCAYO,SAPALLANGA,RECUPERADO_EQUIVALENCIA_VERIFICADA,JUNIN / HUANCAYO / SAPALLANCA => JUNIN / HUANCAYO / SAPALLANGA,NaN,NaN,NaN,NaN
4,PUNO,MELGAR,AYAVARÍ,365,PUNO,MELGAR,AYAVIRI,RECUPERADO_EQUIVALENCIA_VERIFICADA,PUNO / MELGAR / AYAVARI => PUNO / MELGAR / AYAVIRI,NaN,NaN,NaN,NaN
5,ICA,NAZCA,VISTA ALEGRE,324,ICA,NASCA,VISTA ALEGRE,RECUPERADO_EQUIVALENCIA_VERIFICADA,ICA / NAZCA / VISTA ALEGRE => ICA / NASCA / VISTA ALEGRE,NaN,NaN,NaN,NaN
6,CUSCO,QUISPICANCHIS,OROPESA,258,CUSCO,QUISPICANCHI,OROPESA,RECUPERADO_EQUIVALENCIA_VERIFICADA,CUSCO / QUISPICANCHIS / OROPESA => CUSCO / QUISPICANCHI / ORO...,NaN,NaN,NaN,NaN
7,AREQUIPA,CARAVELLÍ,CHALA,248,AREQUIPA,CARAVELI,CHALA,RECUPERADO_EQUIVALENCIA_VERIFICADA,AREQUIPA / CARAVELLI / CHALA => AREQUIPA / CARAVELI / CHALA,NaN,NaN,NaN,NaN
8,CUSCO,QUISPICANCHIS,OCONGATE,242,CUSCO,QUISPICANCHI,OCONGATE,RECUPERADO_EQUIVALENCIA_VERIFICADA,CUSCO / QUISPICANCHIS / OCONGATE => CUSCO / QUISPICANCHI / OC...,NaN,NaN,NaN,NaN
9,CUSCO,QUISPICANCHIS,URCOS,231,CUSCO,QUISPICANCHI,URCOS,RECUPERADO_EQUIVALENCIA_VERIFICADA,CUSCO / QUISPICANCHIS / URCOS => CUSCO / QUISPICANCHI / URCOS,NaN,NaN,NaN,NaN



PRINCIPALES PENDIENTES


,DEP_ORIGEN,PROV_ORIGEN,DIST_ORIGEN,REGISTROS,DEP_INEI_PROPUESTO,PROV_INEI_PROPUESTO,DIST_INEI_PROPUESTO,RESULTADO,EQUIVALENCIA,SUGERENCIA_1,SIMILITUD_1,SUGERENCIA_2,SIMILITUD_2
30,CUSCO,CAÑAS,KUNTURKANNKI,56,CUSCO,CANAS,KUNTURKANNKI,PENDIENTE_REVISION_MANUAL,CUSCO / CANAS / KUNTURKANNKI => CUSCO / CANAS / KUNTURKANNKI,CUSCO/CANAS/KUNTURKANKI,0.957,CUSCO/CANAS/LANGUI,0.333
31,PUNO,PUNO,MAÑOZO,56,PUNO,PUNO,MANOZO,PENDIENTE_REVISION_MANUAL,PUNO / PUNO / MANOZO => PUNO / PUNO / MANOZO,PUNO/PUNO/MANAZO,0.833,PUNO/PUNO/SAN ANTONIO,0.471
40,PUNO,HUANCANÉ,RASASPATA,36,PUNO,HUANCANE,RASASPATA,PENDIENTE_REVISION_MANUAL,PUNO / HUANCANE / RASASPATA => PUNO / HUANCANE / RASASPATA,PUNO/HUANCANE/ROSASPATA,0.889,PUNO/HUANCANE/COJATA,0.400
42,JUNÍN,CHUPACA,SAN JUAN DE YSCOS,33,JUNIN,CHUPACA,SAN JUAN DE YSCOS,PENDIENTE_REVISION_MANUAL,JUNIN / CHUPACA / SAN JUAN DE YSCOS => JUNIN / CHUPACA / SAN ...,JUNIN/CHUPACA/SAN JUAN DE ISCOS,0.941,JUNIN/CHUPACA/SAN JUAN DE JARPA,0.706
44,HUANUCO,HUANUCO,QUISQUI,31,HUANUCO,HUANUCO,QUISQUI,PENDIENTE_REVISION_MANUAL,HUANUCO / HUANUCO / QUISQUI => HUANUCO / HUANUCO / QUISQUI,HUANUCO/HUANUCO/QUISQUI KICHKI,0.667,HUANUCO/HUANUCO/YACUS,0.333
50,AREQUIPA,CAYLLOMA,COPARAQUE,21,AREQUIPA,CAYLLOMA,COPARAQUE,PENDIENTE_REVISION_MANUAL,AREQUIPA / CAYLLOMA / COPARAQUE => AREQUIPA / CAYLLOMA / COPA...,AREQUIPA/CAYLLOMA/COPORAQUE,0.889,AREQUIPA/CAYLLOMA/YANQUE,0.533
52,APURIMAC,GRAU,HUAILLATI,20,APURIMAC,GRAU,HUAILLATI,PENDIENTE_REVISION_MANUAL,APURIMAC / GRAU / HUAILLATI => APURIMAC / GRAU / HUAILLATI,APURIMAC/GRAU/HUAYLLATI,0.889,APURIMAC/GRAU/CHUQUIBAMBILLA,0.609
53,APURIMAC,GRAU,MARISCAL GAMARRA,20,APURIMAC,GRAU,MARISCAL GAMARRA,PENDIENTE_REVISION_MANUAL,APURIMAC / GRAU / MARISCAL GAMARRA => APURIMAC / GRAU / MARIS...,APURIMAC/GRAU/GAMARRA,0.609,APURIMAC/GRAU/MAMARA,0.545
54,ANCASH,CARLOS F FITZCARRALD,SAN LUIS,20,ANCASH,CARLOS F FITZCARRALD,SAN LUIS,PENDIENTE_REVISION_MANUAL,ANCASH / CARLOS F FITZCARRALD / SAN LUIS => ANCASH / CARLOS F...,PROVINCIA_SUGERIDA:ANCASH/CARLOS FERMIN FITZCARRALD,0.889,PROVINCIA_SUGERIDA:ANCASH/ANTONIO RAYMONDI,0.333
59,AYACUCHO,HUAMANGA,HUAMANGA,18,AYACUCHO,HUAMANGA,HUAMANGA,PENDIENTE_REVISION_MANUAL,AYACUCHO / HUAMANGA / HUAMANGA => AYACUCHO / HUAMANGA / HUAMANGA,AYACUCHO/HUAMANGA/CHIARA,0.429,AYACUCHO/HUAMANGA/JESUS NAZARENO,0.364



PASO 13 FINALIZADO


In [24]:
from pathlib import Path
from contextlib import closing
import subprocess
import sqlite3
import sys

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
CARPETA = BASE / "clientes_direcciones"

SCRIPT = BASE / "11_integrar_equivalencias_google_v5_CORREGIDO.py"

DB_ORIGINAL = CARPETA / "direcciones_preparadas_google_masivo.sqlite"
DB_NUEVA = CARPETA / "direcciones_preparadas_google_masivo_v5_inei.sqlite"

ROBOT = BASE / "12_google_maps_clientes_v5_inei.py"

print("=" * 90)
print("PASO 14 CORREGIDO - GOOGLE MAPS V5 + INEI")
print("=" * 90)

# 1. Validar archivos
for nombre, ruta in [
    ("Script corregido", SCRIPT),
    ("SQLite original", DB_ORIGINAL),
]:
    print(f"{nombre}: {'OK' if ruta.is_file() else 'FALTA'}")
    if not ruta.is_file():
        raise FileNotFoundError(ruta)

# 2. Ejecutar la integración corregida
print("\nINICIANDO INTEGRACION...\n")

resultado = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False
)

print(resultado.stdout)

if resultado.stderr:
    print("\nDETALLE:")
    print(resultado.stderr)

if resultado.returncode != 0:
    raise RuntimeError(
        f"Error en Paso 14: {resultado.returncode}"
    )

# 3. Comprobar ambas bases
print("\n" + "=" * 90)
print("VALIDACION DE BASES SQLITE")
print("=" * 90)

for nombre, db in [
    ("ORIGINAL", DB_ORIGINAL),
    ("V5 + INEI", DB_NUEVA),
]:
    if not db.is_file():
        raise FileNotFoundError(db)

    with closing(sqlite3.connect(str(db))) as conn:
        total = conn.execute(
            "SELECT COUNT(*) FROM preparadas"
        ).fetchone()[0]

        integridad = conn.execute(
            "PRAGMA integrity_check"
        ).fetchone()[0]

    print(
        f"{nombre}: {total:,} registros | "
        f"Integridad: {integridad}"
    )

    if total != 159459 or integridad != "ok":
        raise RuntimeError(
            f"La base {nombre} no supero la validacion."
        )

# 4. Verificar robot generado
print("\n" + "=" * 90)
print("VERIFICACION DEL ROBOT V5 + INEI")
print("=" * 90)

if not ROBOT.is_file():
    raise FileNotFoundError(ROBOT)

verificacion = subprocess.run(
    [
        sys.executable, "-u", str(ROBOT),
        "--laptop", "1",
        "--verificar"
    ],
    cwd=str(BASE),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False
)

print(verificacion.stdout)

if verificacion.stderr:
    print(verificacion.stderr)

if verificacion.returncode != 0:
    raise RuntimeError(
        "El robot no supero la verificacion."
    )

print("\n" + "=" * 90)
print("PASO 14 COMPLETADO CORRECTAMENTE")
print("=" * 90)

print("Nueva SQLite:", DB_NUEVA)
print("Robot:", ROBOT)
print("\nListo para comenzar las pruebas Google Maps + INEI.")

PASO 14 CORREGIDO - GOOGLE MAPS V5 + INEI
Script corregido: OK
SQLite original: OK

INICIANDO INTEGRACION...

PASO 14 CORREGIDO - BASE GOOGLE MAPS V5 CON EQUIVALENCIAS INEI
Equivalencias verificadas: 66
Copiando SQLite original en archivo temporal NUEVO...
Copia terminada. Conexiones de respaldo cerradas.
Validación: 159,459 filas; 2,836 corregidas; integridad=ok; journal=delete
Todas las conexiones SQLite cerradas. Publicando archivos...
PASO 14 COMPLETADO
Filas preparadas: 159,459
Direcciones con datos INEI corregidos: 2,836
Combinaciones aplicadas: 64
Nueva base: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\direcciones_preparadas_google_masivo_v5_inei.sqlite
Nuevo robot: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\12_google_maps_clientes_v5_inei.py
Reporte: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\reporte_integracion_inei_v5.csv
Ninguna base histórica fue modificada; no se consultó Google Maps.


VALIDACION DE BASES SQLITE
ORIGINAL: 159,459 registros | Integridad: o

In [27]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import json

BASE = Path(r"D:\API_SUNAT_BK\API_SUNAT\API_MAPS")
CARPETA = BASE / "clientes_direcciones"

SCRIPT = BASE / "13_benchmark_trazabilidad_v5_inei.py"
ROBOT = BASE / "12_google_maps_clientes_v5_inei.py"

# CONFIGURACION DE LA PRUEBA
DESDE = 31
HASTA = 300
MUESTRA = 30
LAPTOP = 1

print("=" * 90)
print("BENCHMARK GOOGLE MAPS - AUDITORIA DE TIEMPOS Y CONSULTAS")
print("=" * 90)

for nombre, ruta in [
    ("Benchmark", SCRIPT),
    ("Robot V5 + INEI", ROBOT)
]:
    print(f"{nombre}: {'OK' if ruta.is_file() else 'FALTA'}")
    if not ruta.is_file():
        raise FileNotFoundError(ruta)

print(f"\nMuestra: {MUESTRA} direcciones nuevas")
print(f"Rango: {DESDE} - {HASTA}")
print("Registrando cada consulta a Google Maps...\n")

comando = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--laptop", str(LAPTOP),
    "--desde", str(DESDE),
    "--hasta", str(HASTA),
    "--max-filas", str(MUESTRA)
]

proceso = subprocess.Popen(
    comando,
    cwd=str(BASE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

try:
    for linea in proceso.stdout:
        print(linea, end="", flush=True)
    codigo = proceso.wait()
except KeyboardInterrupt:
    proceso.terminate()
    proceso.wait()
    print("\nPrueba interrumpida.")
    raise

print("\nCódigo de salida:", codigo)

# Recuperar los archivos de la última ejecución
DIRECTORIO = CARPETA / "benchmarks_trazabilidad"

corridas = sorted(
    DIRECTORIO.glob("corrida_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not corridas:
    raise RuntimeError("No se encontraron archivos del benchmark.")

SALIDA = corridas[0]

print("\n" + "=" * 90)
print("ARCHIVOS Y ANALISIS DE CONSULTAS")
print("=" * 90)

print("Carpeta de resultados:", SALIDA)

archivo = SALIDA / "traza_consultas_detalle.csv"

if archivo.exists():

    df = pd.read_csv(
        archivo,
        sep=";",
        encoding="utf-8-sig"
    )

    print(f"\nTotal de consultas registradas: {len(df):,}")

    columnas = [
        "FILA_ID",
        "N_CONSULTA",
        "TIPO_CONSULTA",
        "QUERY",
        "SEGUNDOS",
        "FICHAS_ABIERTAS",
        "RESULTADOS_BRUTOS",
        "RESULTADOS_VALIDOS",
        "ELEGIDA_FINAL",
        "RESULTADO_QUERY"
    ]

    pd.set_option("display.max_colwidth", 100)
    pd.set_option("display.max_columns", None)

    print("\nDETALLE DE LAS PRIMERAS 30 CONSULTAS")
    display(df[columnas].head(30))

    print("\nLAS 20 CONSULTAS MAS LENTAS")
    display(
        df[columnas]
        .sort_values("SEGUNDOS", ascending=False)
        .head(20)
    )

    print("\nTIEMPO POR TIPO DE BUSQUEDA")
    display(
        pd.read_csv(
            SALIDA / "resumen_tipos_consulta.csv",
            sep=";",
            encoding="utf-8-sig"
        )
    )

    print("\nEFICIENCIA SEGUN ORDEN DE CONSULTA")
    display(
        pd.read_csv(
            SALIDA / "resumen_orden_consulta.csv",
            sep=";",
            encoding="utf-8-sig"
        )
    )

    print("\nTIEMPO POR DIRECCION")
    display(
        pd.read_csv(
            SALIDA / "tiempos_por_direccion.csv",
            sep=";",
            encoding="utf-8-sig"
        )
    )

else:
    print("No se encontraron consultas registradas.")

print("\n" + "=" * 90)
print("BENCHMARK FINALIZADO")
print("=" * 90)

BENCHMARK GOOGLE MAPS - AUDITORIA DE TIEMPOS Y CONSULTAS
Benchmark: OK
Robot V5 + INEI: OK

Muestra: 30 direcciones nuevas
Rango: 31 - 300
Registrando cada consulta a Google Maps...

BENCHMARK DE CONSULTAS REALES - GOOGLE MAPS V5 + INEI
Robot: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\12_google_maps_clientes_v5_inei.py
Rango: 31-300 | Máximo de direcciones NUEVAS: 30
Salida local: D:\API_SUNAT_BK\API_SUNAT\API_MAPS\clientes_direcciones\benchmarks_trazabilidad\corrida_20261116_183814
No se cambia el algoritmo ni las SQLite históricas.
CAPTCHA      : se detiene; NO se evade
FILA 32 | RUC 20102911 | SOLO_VIA | PSJ OROVILCA MZ C URB SOL DE ICA ICA
    [TRAZA_Q] Fila=32 Q=1 tiempo=7.6s fichas=0 candidatos=1 estado=OK
    [TRAZA_Q] Fila=32 Q=2 tiempo=20.6s fichas=4 candidatos=4 estado=OK
    [TRAZA_Q] Fila=32 Q=3 tiempo=6.4s fichas=0 candidatos=1 estado=OK
    [TRAZA_Q] Fila=32 Q=4 tiempo=6.1s fichas=0 candidatos=1 estado=OK
    FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_ZONA | -14.0648602, -75.72

,FILA_ID,N_CONSULTA,TIPO_CONSULTA,QUERY,SEGUNDOS,FICHAS_ABIERTAS,RESULTADOS_BRUTOS,RESULTADOS_VALIDOS,ELEGIDA_FINAL,RESULTADO_QUERY
0,32,1,ORIGINAL,"PSJ OROVILCA MZ C URB SOL DE ICA ICA, ICA, PERU",7.561,0,1,0,0,SOLO_RECHAZADOS
1,32,2,VIA_ZONA,"PASAJE OROVILCA MZ C, SOL DE ICA ICA, ICA, PERU",20.625,4,4,0,0,SOLO_RECHAZADOS
2,32,3,SOLO_VIA,"PASAJE OROVILCA MZ C, ICA, PERU",6.365,0,1,0,0,SOLO_RECHAZADOS
3,32,4,SOLO_ZONA,"SOL DE ICA ICA, ICA, PERU",6.079,0,1,1,1,COORDENADA_ELEGIDA
4,33,1,ORIGINAL,"JR LORO MICUNA MZ B3 CALLERIA, CALLERIA, CORONEL PORTILLO, UCAYALI, PERU",6.069,0,1,0,0,SOLO_RECHAZADOS
5,33,2,SOLO_VIA,"JIRON LORO MICUNA MZ B3 CALLERIA, CALLERIA, CORONEL PORTILLO, UCAYALI, PERU",10.967,2,2,0,0,SOLO_RECHAZADOS
6,34,1,ORIGINAL,"JR MIRAFLORES URB MIRAFLORES PERENE, PERENE, CHANCHAMAYO, JUNIN, PERU",6.102,0,1,0,0,SOLO_RECHAZADOS
7,34,2,VIA_ZONA,"JIRON MIRAFLORES, MIRAFLORES PERENE, PERENE, CHANCHAMAYO, JUNIN, PERU",6.046,0,1,0,0,SOLO_RECHAZADOS
8,34,3,SOLO_VIA,"JIRON MIRAFLORES, PERENE, CHANCHAMAYO, JUNIN, PERU",6.036,0,1,0,0,SOLO_RECHAZADOS
9,34,4,SOLO_ZONA,"MIRAFLORES PERENE, PERENE, CHANCHAMAYO, JUNIN, PERU",10.566,2,2,0,0,SOLO_RECHAZADOS



LAS 20 CONSULTAS MAS LENTAS


,FILA_ID,N_CONSULTA,TIPO_CONSULTA,QUERY,SEGUNDOS,FICHAS_ABIERTAS,RESULTADOS_BRUTOS,RESULTADOS_VALIDOS,ELEGIDA_FINAL,RESULTADO_QUERY
1,32,2,VIA_ZONA,"PASAJE OROVILCA MZ C, SOL DE ICA ICA, ICA, PERU",20.625,4,4,0,0,SOLO_RECHAZADOS
69,53,4,SOLO_ZONA,"A UNA CUADRA JULIACA, JULIACA, SAN ROMAN, PUNO, PERU",15.944,4,4,2,1,COORDENADA_ELEGIDA
77,58,1,ORIGINAL,"AV MIGUEL GRAU 235 URB BARRIO LA COLMENA CAJAMARCA, CAJAMARCA, PERU",15.734,4,4,1,0,CON_EVIDENCIA
39,45,2,VIA_ZONA,"CALLE JOSE OLAYA MZ G, VIUDA DE CANEVARO SAN JUAN DE MIRAFLORES, SAN JUAN DE MIRAFLORES, LIMA, PERU",15.702,4,4,3,1,COORDENADA_ELEGIDA
14,36,3,SOLO_VIA,"AVENIDA REPUBLICA DE PANAMA 3055 SAN ISIDRO, MOROCOCHA, YAULI, JUNIN, PERU",15.696,4,4,0,0,SOLO_RECHAZADOS
75,57,1,ORIGINAL,"JR GRAU 1001 YUCAY, YUCAY, URUBAMBA, CUSCO, PERU",15.647,4,4,0,0,SOLO_RECHAZADOS
40,45,3,SOLO_VIA,"CALLE JOSE OLAYA MZ G, SAN JUAN DE MIRAFLORES, LIMA, PERU",15.577,4,4,1,0,CON_EVIDENCIA
62,52,1,ORIGINAL,"JR CARABAYA 836 URB MANCO CAPAC JULIACA, JULIACA, SAN ROMAN, PUNO, PERU",15.490,4,4,1,0,CON_EVIDENCIA
36,44,1,ORIGINAL,"JR CONSTITUCION 940 HUANUCO, HUANUCO, PERU",15.467,4,4,3,0,CON_EVIDENCIA
51,48,4,SOLO_ZONA,"7 MA ZONA EL AGUSTINO, EL AGUSTINO, LIMA, PERU",15.433,4,4,0,0,SOLO_RECHAZADOS



TIEMPO POR TIPO DE BUSQUEDA


,TIPO_CONSULTA,CONSULTAS,SEGUNDOS_TOTALES,SEGUNDOS_PROMEDIO,FICHAS_ABIERTAS,CONSULTAS_CON_EVIDENCIA,CONSULTAS_ELEGIDAS
0,ORIGINAL,30,321.338,10.711267,61,13,8
1,SOLO_VIA,25,205.116,8.204640,26,11,6
2,VIA_ZONA,16,177.263,11.078938,34,6,3
3,SOLO_ZONA,17,157.294,9.252588,24,11,5



EFICIENCIA SEGUN ORDEN DE CONSULTA


,N_CONSULTA,CONSULTAS,SEGUNDOS_TOTALES,SEGUNDOS_PROMEDIO,CONSULTAS_CON_EVIDENCIA,CONSULTAS_ELEGIDAS
0,1,30,321.338,10.711267,13,8
1,2,27,255.553,9.464926,13,7
2,3,16,138.410,8.650625,6,2
3,4,15,145.710,9.714000,9,5



TIEMPO POR DIRECCION


,FILA_ID,SEGUNDOS_FILA_APROX,FINALIZACION
0,32,40.658,"FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_ZONA | -14.0648602, -75.728274"
1,33,17.047,FINAL -> SIN_UBICACION | SIN_MATCH | SIN COORDENADAS
2,34,28.765,FINAL -> SIN_UBICACION | SIN_MATCH | SIN COORDENADAS
3,35,16.174,"FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_ZONA | -11.9847957, -76.8155113"
4,36,61.192,FINAL -> SIN_UBICACION | SIN_MATCH | SIN COORDENADAS
5,37,39.763,"FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_ZONA | -12.0201464, -76.8175454"
6,38,8.234,"FINAL -> ENCONTRADO_REFERENCIA | COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR | -12.0239092, -77.0260272"
7,39,20.232,"FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_VIA_POR_VERIFICAR | -14.0670846, -75.7236071"
8,40,17.507,"FINAL -> ENCONTRADO_REFERENCIA | COINCIDENCIA_VIA_NUMERO_POR_VERIFICAR | -12.1570153, -76.9731585"
9,41,30.362,"FINAL -> ENCONTRADO_REFERENCIA | REFERENCIA_ZONA | -14.0707269, -75.7375124"



BENCHMARK FINALIZADO
